Synthetic data generation

In [ ]:
import os
import csv
import glob
import numpy as np
import openlpt as lpt

# --- Settings ---
cam_dir = r"H:\20260210\T0\Round1\camFile"
out_csv = r"H:\20260210\T0\synthetic_wand_points.csv"

N_pairs = 2500
wand_len = 10.0
R_small = 1.5  # mm
R_large = 2.0  # mm

# --- Load all cameras ---
cam_files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
if not cam_files:
    raise FileNotFoundError(f"No cam*.txt found in {cam_dir}")

cams = {}
for path in cam_files:
    name = os.path.basename(path)
    # Extract integer id from camX.txt
    cid_str = name.replace("cam", "").replace(".txt", "")
    cid = int(cid_str)
    cam = lpt.Camera()
    cam.loadParameters(path)
    cams[cid] = cam

cam_ids = sorted(cams.keys())
print("Loaded cameras:", cam_ids)

# --- Helper: project point and compute pixel radius ---
def estimate_radius_px(cam, X, R_mm):
    """
    Estimate pixel radius for a 3D sphere of radius R_mm at X
    using finite differences and cam.project only.
    Returns NaN if projection fails.
    """
    X = np.asarray(X, dtype=float)

    # Project center
    ptC = lpt.Pt3D(float(X[0]), float(X[1]), float(X[2]))
    uvC = cam.project(ptC)
    u0, v0 = float(uvC[0]), float(uvC[1])
    if not np.isfinite(u0) or not np.isfinite(v0):
        return np.nan

    # Use camera line-of-sight direction to build tangent basis
    # Approx: dir = (X - C), where C from pinplate param
    pp = cam._pinplate_param
    C = np.array([pp.t_vec_inv[0], pp.t_vec_inv[1], pp.t_vec_inv[2]], dtype=float)
    dir_vec = X - C
    norm = np.linalg.norm(dir_vec)
    if not np.isfinite(norm) or norm < 1e-9:
        return np.nan
    dir_vec /= norm

    # Build two orthonormal tangents
    tmp = np.array([1.0, 0.0, 0.0]) if abs(dir_vec[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    t1 = np.cross(dir_vec, tmp); t1 /= np.linalg.norm(t1)
    t2 = np.cross(dir_vec, t1)

    # Project ±R along tangents
    def proj_offset(t):
        Xp = X + R_mm * t
        Xm = X - R_mm * t
        up = cam.project(lpt.Pt3D(float(Xp[0]), float(Xp[1]), float(Xp[2])))
        um = cam.project(lpt.Pt3D(float(Xm[0]), float(Xm[1]), float(Xm[2])))
        up = np.array([float(up[0]), float(up[1])], dtype=float)
        um = np.array([float(um[0]), float(um[1])], dtype=float)
        if np.any(~np.isfinite(up)) or np.any(~np.isfinite(um)):
            return np.nan
        return 0.5 * np.linalg.norm(up - um)

    r1 = proj_offset(t1)
    r2 = proj_offset(t2)

    if not np.isfinite(r1) and not np.isfinite(r2):
        return np.nan
    if not np.isfinite(r1):
        return r2
    if not np.isfinite(r2):
        return r1
    return 0.5 * (r1 + r2)

def project_with_radius(cam, X, R_mm):
    pt = lpt.Pt3D(float(X[0]), float(X[1]), float(X[2]))
    uv = cam.project(pt)
    u = float(uv[0])
    v = float(uv[1])
    if not np.isfinite(u) or not np.isfinite(v):
        return None
    r_px = estimate_radius_px(cam, X, R_mm)
    if not np.isfinite(r_px):
        return None
    return (u, v, float(r_px))

# --- Generate valid wand pairs ---
rows = []
frame_id = 0

rng = np.random.default_rng(42)

def random_unit_vector():
    v = rng.normal(size=3)
    v /= np.linalg.norm(v)
    return v

while frame_id < N_pairs:
    # random center within cube, but ensure endpoints stay inside [-10,10]
    # use center in [-5,5] so ±5 stays inside
    C = rng.uniform(-5.0, 5.0, size=3)
    d = random_unit_vector()
    A = C - 0.5 * wand_len * d
    B = C + 0.5 * wand_len * d

    # ensure inside 20x20x20 cube centered at origin
    if np.any(A < -10) or np.any(A > 10) or np.any(B < -10) or np.any(B > 10):
        continue

    # Project for all cams; if any cam fails, reject this pair
    projections = {}
    valid = True
    for cid in cam_ids:
        cam = cams[cid]
        proj_s = project_with_radius(cam, A, R_small)
        proj_l = project_with_radius(cam, B, R_large)
        if (proj_s is None) or (proj_l is None):
            valid = False
            break
        projections[cid] = (proj_s, proj_l)

    if not valid:
        continue    

    # Store rows: Filtered_Small (PointIdx=0), Filtered_Large (PointIdx=1)
    for cid in cam_ids:
        (u_s, v_s, r_s), (u_l, v_l, r_l) = projections[cid]
        rows.append([frame_id, cid, "Filtered_Small", 0, f"{u_s:.3f}", f"{v_s:.3f}", f"{r_s:.3f}", "0"])
        rows.append([frame_id, cid, "Filtered_Large", 1, f"{u_l:.3f}", f"{v_l:.3f}", f"{r_l:.3f}", "0"])

    frame_id += 1
    if frame_id % 100 == 0:
        print(f"Generated frame {frame_id}/{N_pairs}", end="\r")

print("Generated frames:", frame_id)
print("Total rows:", len(rows))

# --- Write CSV ---
with open(out_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Frame", "Camera", "Status", "PointIdx", "X", "Y", "Radius", "Metric"])
    writer.writerows(rows)

print("Saved:", out_csv)


Given true refraction plane, check camera calibration

In [1]:
import os
import re
import csv
import glob
import numpy as np
import cv2

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAOptimizer,
    RefractiveBAConfig,
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import (
    CppCameraFactory,
)

# -----------------------------
# Paths
# -----------------------------
truth_cam_dir = r"H:\20260210\T0\Round1\camFile"          # 真值平面/介质来源
init_cam_dir  = r"H:\20260210\T0\Synthetic_test\camFile"  # 相机初值来源
out_csv       = r"H:\20260210\T0\synthetic_wand_points.csv"

# 可选：为了先快速验证，限制帧数；想全量就设为 0
MAX_FRAMES = 300

# -----------------------------
# Helpers
# -----------------------------
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and not s.startswith("#"):
            return s, j
        j += 1
    return "", j

def parse_cam_file(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    d = {}
    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix") or s.startswith("# Camera Matrix K"):
            row, _ = _next_data(lines, i)
            d["f"] = float(row.split()[0])

        elif s.startswith("# Distortion"):
            row, _ = _next_data(lines, i)
            vals = [float(x) for x in row.split(",")]
            d["k1"] = vals[0] if len(vals) > 0 else 0.0
            d["k2"] = vals[1] if len(vals) > 1 else 0.0

        elif s.startswith("# Rotation Matrix R") or s.startswith("# Rotation Matrix:"):
            r1, _ = _next_data(lines, i)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            R = np.array(
                [
                    [float(x) for x in r1.split()],
                    [float(x) for x in r2.split()],
                    [float(x) for x in r3.split()],
                ],
                dtype=float,
            )
            d["R"] = R

        elif s.startswith("# Translation Vector T") or s.startswith("# Translation Vector:"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "WINDOW_ID=" in s:
            d["wid"] = int(s.split("WINDOW_ID=")[1])

        elif "plane.pt" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "plane.norm_vector" in s:
            row, _ = _next_data(lines, i)
            n = np.array([float(x) for x in row.split()], dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = float(row.split(",")[0])

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = [float(x) for x in row.split(",")]
            # file: obj,win,air -> n3,n2,n1
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    d["cx"] = 640.0
    d["cy"] = 400.0
    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params = {}
    cam_to_window = {}
    window_planes = {}
    window_media = {}

    files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    if not files:
        raise FileNotFoundError(f"No cam*.txt in {cam_dir}")

    for path in files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array(
            [
                d["rvec"][0], d["rvec"][1], d["rvec"][2],
                d["tvec"][0], d["tvec"][1], d["tvec"][2],
                d["f"], d["cx"], d["cy"],
                d.get("k1", 0.0), d.get("k2", 0.0),
            ],
            dtype=np.float64,
        )

        cam_to_window[cid] = d["wid"]

        if with_plane and d["wid"] not in window_planes:
            # 文件里是 farthest interface，优化内部使用 closest interface
            p_closest = d["plane_far"] - d["plane_n"] * d["thickness"]
            wid = d["wid"]
            window_planes[wid] = {
                "plane_pt": p_closest,
                "plane_n": d["plane_n"],
                "initialized": True,
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"],
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs_csv(csv_path):
    obsA, obsB = {}, {}
    frames = set()

    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    return obsA, obsB, frames

def rot_err_deg(rvec_est, rvec_gt):
    R1, _ = cv2.Rodrigues(rvec_est)
    R2, _ = cv2.Rodrigues(rvec_gt)
    R = R2 @ R1.T
    c = (np.trace(R) - 1.0) / 2.0
    c = max(-1.0, min(1.0, c))
    return float(np.degrees(np.arccos(c)))

# -----------------------------
# Build test setup
# -----------------------------
cam_gt, map_gt, plane_gt, media_gt = load_cam_set(truth_cam_dir, with_plane=True)
cam_init, map_init, _, _ = load_cam_set(init_cam_dir, with_plane=False)

if map_gt != map_init:
    raise RuntimeError("cam_to_window mismatch between truth and init cam folders")

obsA, obsB, frames_all = load_obs_csv(out_csv)
if MAX_FRAMES > 0:
    frames = frames_all[:MAX_FRAMES]
else:
    frames = frames_all

dataset = {
    "obsA": obsA,
    "obsB": obsB,
    "frames": frames,
    "cam_ids": sorted(cam_init.keys()),
    "est_radius_small_mm": 1.5,
    "est_radius_large_mm": 2.0,
}

class _Base:
    pass

base = _Base()
base.image_size = (800, 1280)

# 用“初始相机 + 真值平面/介质”初始化 C++ 相机
cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=media_gt,
    cam_to_window=map_gt,
    window_planes=plane_gt,
)

# -----------------------------
# Progress callback (every nfev=10)
# -----------------------------
cb_count = {"k": 0}
def progress_cb(phase, ray_rmse, len_rmse, cost):
    cb_count["k"] += 1
    # BA内部已改成每10次residual callback一次，这里按nfev=10递增显示
    nfev_est = cb_count["k"] * 10
    print(
        f"[nfev~{nfev_est:4d}] {phase} | "
        f"ray_rmse={ray_rmse:.6f} mm | len_rmse={len_rmse:.6f} mm | cost={cost:.6e}"
    )

cfg = RefractiveBAConfig(
    stage=0,          # 我们手动调用 _optimize_generic
    verbosity=1,      # 可看到更多日志
    max_frames=MAX_FRAMES if MAX_FRAMES > 0 else 0,
)

opt = RefractiveBAOptimizer(
    dataset=dataset,
    cam_params=cam_init,        # 初值（Synthetic_test）
    cams_cpp=cams_cpp,
    cam_to_window=map_gt,
    window_media=media_gt,      # 真值介质
    window_planes=plane_gt,     # 真值平面
    wand_length=10.0,
    config=cfg,
    progress_callback=progress_cb,
)

opt._compute_physical_sigmas()

print("\n=== Before (vs GT) ===")
for cid in sorted(cam_init.keys()):
    e_t = np.linalg.norm(cam_init[cid][3:6] - cam_gt[cid][3:6])
    e_r = rot_err_deg(cam_init[cid][0:3], cam_gt[cid][0:3])
    print(f"Cam {cid}: t_err={e_t:.6f} mm, r_err={e_r:.6f} deg")

# -----------------------------
# Camera-only optimization (planes fixed to truth)
# -----------------------------
opt._optimize_generic(
    mode="cam_only_truth_plane",
    description="CamOnly TruthPlane",
    enable_planes=False,         # 只优化相机
    enable_cam_t=True,
    enable_cam_r=True,
    limit_rot_rad=np.radians(20.0),
    limit_trans_mm=50.0,
    limit_plane_d_mm=0.0,
    limit_plane_angle_rad=0.0,
    ftol=1e-8,
    loss="linear",
)

print("\n=== After (vs GT) ===")
for cid in sorted(cam_init.keys()):
    p = opt.cam_params[cid]
    e_t = np.linalg.norm(p[3:6] - cam_gt[cid][3:6])
    e_r = rot_err_deg(p[0:3], cam_gt[cid][0:3])
    print(f"Cam {cid}: t_err={e_t:.6f} mm, r_err={e_r:.6f} deg")

lam = 2.0 * max(1, len(cam_init))
_, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
    opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
)
ray_rmse = np.sqrt(S_ray / max(1, N_ray))
len_rmse = np.sqrt(S_len / max(1, N_len)) if N_len > 0 else 0.0

print("\n=== Final residuals ===")
print(f"frames={len(frames)}, N_ray={N_ray}, N_len={N_len}")
print(f"ray_rmse_mm={ray_rmse:.9f}")
print(f"len_rmse_mm={len_rmse:.9f}")



Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
Unit Normalization: sigma_ray=0.0114mm (0.5px at Z=206.0mm), sigma_wand=0.2000mm (2.0%)

=== Before (vs GT) ===
Cam 0: t_err=0.697742 mm, r_err=2.344519 deg
Cam 1: t_err=1.100389 mm, r_err=1.558594 deg
Cam 2: t_err=0.667578 mm, r_err=1.632894 deg
Cam 3: t_err=0.525099 mm, r_err=1.833747 deg
  diff_step: rvec=0.0001, tvec=0.01, plane_d=0.01, plane_ang=0.0001, f=0.1, thick=0.001, k=1e-06
  [CamOnly TruthPlane] optimizing 24 parameters (8 blocks)...
    Global Fixed Weighting: lambda=8.0 (N_cams=4)
    Initial: S_ray=40.21, S_len=0.20 (J0=0.0222)
[nfev~  10] CamOnly TruthPlane | ray_rmse=0.071562 mm | len_rmse=0.025226 mm | cost=4.695107e+04
[nfev~  20] CamOnly TruthPlane | ray_rmse=0.059677 mm | len_rmse=0.022063 mm | cost=3.265260e+04
[nfev~  30] CamOnly TruthPlane | ray_rmse=0.043556 mm | len_rmse=0.019343 mm | cost=1.739737e+04
[nfev~  40] CamOnly TruthPlane | ray_rmse=0.027230 mm | len_rmse=0.016464 mm 

Add bias to plane location to check camera calibration

In [2]:
import os
import re
import csv
import glob
import copy
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAOptimizer,
    RefractiveBAConfig,
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import (
    CppCameraFactory,
)

# =========================
# Paths / Settings
# =========================
truth_cam_dir = r"H:\20260210\T0\Round1\camFile"          # 真值平面
init_cam_dir  = r"H:\20260210\T0\Synthetic_test\camFile"  # 相机初值
obs_csv       = r"H:\20260210\T0\synthetic_wand_points.csv"

WAND_LEN_MM = 10.0
R_SMALL_MM = 1.5
R_LARGE_MM = 2.0
MAX_FRAMES = 300          # 0 表示全量
BIAS_LIST_MM = np.arange(-15.0, 15.01, 2.5)  # 你问的偏差设置
SHOW_PROGRESS = True      # True: 每10个nfev打印一次

# =========================
# Helpers
# =========================
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and not s.startswith("#"):
            return s, j
        j += 1
    return "", j

def parse_cam_file(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    d = {}
    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix") or s.startswith("# Camera Matrix K"):
            row, _ = _next_data(lines, i)
            d["f"] = float(row.split()[0])

        elif s.startswith("# Distortion"):
            row, _ = _next_data(lines, i)
            vals = [float(x) for x in row.split(",")]
            d["k1"] = vals[0] if len(vals) > 0 else 0.0
            d["k2"] = vals[1] if len(vals) > 1 else 0.0

        elif s.startswith("# Rotation Matrix R") or s.startswith("# Rotation Matrix:"):
            r1, _ = _next_data(lines, i)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array(
                [[float(x) for x in r1.split()],
                 [float(x) for x in r2.split()],
                 [float(x) for x in r3.split()]],
                dtype=float
            )

        elif s.startswith("# Translation Vector T") or s.startswith("# Translation Vector:"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "WINDOW_ID=" in s:
            d["wid"] = int(s.split("WINDOW_ID=")[1])

        elif "plane.pt" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "plane.norm_vector" in s:
            row, _ = _next_data(lines, i)
            n = np.array([float(x) for x in row.split()], dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = float(row.split(",")[0])

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = [float(x) for x in row.split(",")]  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    d["cx"] = 640.0
    d["cy"] = 400.0
    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params = {}
    cam_to_window = {}
    window_planes = {}
    window_media = {}

    files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    if not files:
        raise FileNotFoundError(f"No cam*.txt found in {cam_dir}")

    for path in files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array(
            [
                d["rvec"][0], d["rvec"][1], d["rvec"][2],
                d["tvec"][0], d["tvec"][1], d["tvec"][2],
                d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0),
            ],
            dtype=np.float64,
        )
        cam_to_window[cid] = d["wid"]

        if with_plane and d["wid"] not in window_planes:
            # 文件中是 farthest interface；优化内部用 closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            wid = d["wid"]
            window_planes[wid] = {
                "plane_pt": p_close,
                "plane_n": d["plane_n"],
                "initialized": True,
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"],
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB = {}, {}
    frames = set()

    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA,
        "obsB": obsB,
        "frames": frames,
        "cam_ids": None,  # later填
        "est_radius_small_mm": R_SMALL_MM,
        "est_radius_large_mm": R_LARGE_MM,
    }

def rot_err_deg(rvec_est, rvec_gt):
    R1, _ = cv2.Rodrigues(rvec_est)
    R2, _ = cv2.Rodrigues(rvec_gt)
    R = R2 @ R1.T
    c = (np.trace(R) - 1.0) / 2.0
    c = np.clip(c, -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))

def mean_cam_err(cam_est, cam_gt):
    t_err = []
    r_err = []
    for cid in sorted(cam_gt.keys()):
        t_err.append(np.linalg.norm(cam_est[cid][3:6] - cam_gt[cid][3:6]))
        r_err.append(rot_err_deg(cam_est[cid][0:3], cam_gt[cid][0:3]))
    return float(np.mean(t_err)), float(np.mean(r_err))

def apply_plane_bias(truth_planes, bias_mm):
    """沿每个window法向将plane_pt(closest)平移 bias_mm。"""
    planes = copy.deepcopy(truth_planes)
    for wid, pl in planes.items():
        n = np.asarray(pl["plane_n"], dtype=float)
        planes[wid]["plane_pt"] = np.asarray(pl["plane_pt"], dtype=float) + bias_mm * n
        planes[wid]["initialized"] = True
    return planes

# =========================
# Load baseline data
# =========================
cam_truth, cam2win_truth, plane_truth, media_truth = load_cam_set(truth_cam_dir, with_plane=True)
cam_init_base, cam2win_init, _, _ = load_cam_set(init_cam_dir, with_plane=False)
if cam2win_truth != cam2win_init:
    raise RuntimeError("cam_to_window mismatch between truth and init")

dataset = load_obs_dataset(obs_csv, max_frames=MAX_FRAMES)
dataset["cam_ids"] = sorted(cam_init_base.keys())

print("Cams:", sorted(cam_init_base.keys()))
print("Frames used:", len(dataset["frames"]))
print("Bias list (mm):", BIAS_LIST_MM)

# =========================
# Experiment A
# =========================
records = []

for bias in BIAS_LIST_MM:
    print(f"\n=== Bias {bias:+.2f} mm ===")

    # 相机初值每次重置
    cam_init = {cid: p.copy() for cid, p in cam_init_base.items()}

    # 真值平面加偏差
    plane_biased = apply_plane_bias(plane_truth, bias)

    # C++ cams: 初始相机 + 偏置平面 + 真值介质
    class _Base:
        pass
    base = _Base()
    base.image_size = (800, 1280)

    cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
        base=base,
        cam_params=cam_init,
        window_media=media_truth,
        cam_to_window=cam2win_truth,
        window_planes=plane_biased,
    )

    # progress callback: 每10个nfev
    cb_counter = {"k": 0}
    def progress_cb(phase, ray_rmse, len_rmse, cost):
        if not SHOW_PROGRESS:
            return
        cb_counter["k"] += 1
        nfev_est = cb_counter["k"] * 10
        print(
            f"[nfev~{nfev_est:4d}] {phase} | "
            f"ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | cost={cost:.6e}"
        )

    cfg = RefractiveBAConfig(
        stage=0,
        verbosity=0,
        max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0),
    )

    opt = RefractiveBAOptimizer(
        dataset=dataset,
        cam_params=cam_init,
        cams_cpp=cams_cpp,
        cam_to_window=cam2win_truth,
        window_media=media_truth,
        window_planes=plane_biased,
        wand_length=WAND_LEN_MM,
        config=cfg,
        progress_callback=progress_cb,
    )
    opt._compute_physical_sigmas()

    # A: 只优化相机参数，平面固定
    opt._optimize_generic(
        mode="expA_cam_only",
        description="ExpA CamOnly",
        enable_planes=False,
        enable_cam_t=True,
        enable_cam_r=True,
        limit_rot_rad=np.radians(20.0),
        limit_trans_mm=50.0,
        limit_plane_d_mm=0.0,
        limit_plane_angle_rad=0.0,
        ftol=1e-7,
        loss="linear",
    )

    lam = 2.0 * max(1, len(cam_init))
    _, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
        opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
    )
    ray_rmse = np.sqrt(S_ray / max(1, N_ray))
    len_rmse = np.sqrt(S_len / max(1, N_len)) if N_len > 0 else np.nan

    cam_t_err_mean, cam_r_err_mean = mean_cam_err(opt.cam_params, cam_truth)

    records.append({
        "bias_mm": float(bias),
        "ray_rmse_mm": float(ray_rmse),
        "len_rmse_mm": float(len_rmse),
        "cam_t_err_mean_mm": cam_t_err_mean,
        "cam_r_err_mean_deg": cam_r_err_mean,
        "N_ray": int(N_ray),
        "N_len": int(N_len),
    })

dfA = pd.DataFrame(records).sort_values("bias_mm").reset_index(drop=True)
display(dfA)

plt.figure(figsize=(7,4))
plt.plot(dfA["bias_mm"], dfA["ray_rmse_mm"], marker="o")
plt.xlabel("Plane bias along normal (mm)")
plt.ylabel("Ray RMSE (mm)")
plt.title("Experiment A: Plane Bias vs Ray RMSE (Camera-only optimize)")
plt.grid(True, alpha=0.3)
plt.show()


Cams: [0, 1, 2, 3]
Frames used: 300
Bias list (mm): [-15.  -12.5 -10.   -7.5  -5.   -2.5   0.    2.5   5.    7.5  10.   12.5
  15. ]

=== Bias -15.00 mm ===

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
  [ExpA CamOnly] optimizing 24 parameters (8 blocks)...
    Global Fixed Weighting: lambda=8.0 (N_cams=4)
    Initial: S_ray=1522.64, S_len=10.87 (J0=0.9243)
[nfev~  10] ExpA CamOnly | ray=0.706828 mm | len=0.187868 mm | cost=4.572865e+06
[nfev~  20] ExpA CamOnly | ray=0.528830 mm | len=0.183165 mm | cost=2.560134e+06
[nfev~  30] ExpA CamOnly | ray=0.182582 mm | len=0.174634 mm | cost=3.059692e+05
[nfev~  40] ExpA CamOnly | ray=0.038963 mm | len=0.152343 mm | cost=1.458809e+04
[nfev~  50] ExpA CamOnly | ray=0.009081 mm | len=0.049766 mm | cost=8.289917e+02
[nfev~  60] ExpA CamOnly | ray=0.004865 mm | len=0.010948 mm | cost=2.202025e+02
[nfev~  70] ExpA CamOnly | ray=0.003778 mm | len=0.010940 mm | cost=1.341836e+02
[nfev~  80] ExpA CamOnly | ray=0.003

,bias_mm,ray_rmse_mm,len_rmse_mm,cam_t_err_mean_mm,cam_r_err_mean_deg,N_ray,N_len
0,-15.0,0.003779,0.010941,3.288802,1.020728,2400,300
1,-12.5,0.003100,0.007442,2.823483,1.026195,2400,300
2,-10.0,0.001975,0.005695,2.240727,0.752024,2400,300
3,-7.5,0.001690,0.004156,1.687466,0.547158,2400,300
4,-5.0,0.002006,0.003612,1.106522,0.671138,2400,300
5,-2.5,0.002294,0.004428,0.544683,0.679791,2400,300
6,0.0,0.001241,0.001110,0.205398,0.363705,2400,300
7,2.5,0.016832,0.051525,2.195331,3.747433,2400,300
8,5.0,0.029692,0.183603,6.442344,5.731180,2400,300
9,7.5,0.025168,0.103136,7.576402,4.513467,2400,300


add bias to plane location, calibrate camera and palen

In [3]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 复用你实验A里已经定义好的这些对象/函数：
# - cam_truth, cam_init_base
# - cam2win_truth, plane_truth, media_truth
# - dataset
# - apply_plane_bias(...)
# - mean_cam_err(...)
# - RefractiveBAConfig, RefractiveBAOptimizer, CppCameraFactory

BIAS_LIST_MM = np.arange(-15.0, 15.01, 2.5)
SHOW_PROGRESS = True  # True时每10个nfev打印一次

records_B = []

for bias in BIAS_LIST_MM:
    print(f"\n=== [ExpB] Bias {bias:+.2f} mm ===")

    cam_init = {cid: p.copy() for cid, p in cam_init_base.items()}
    plane_biased = apply_plane_bias(plane_truth, bias)

    class _Base:
        pass
    base = _Base()
    base.image_size = (800, 1280)

    cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
        base=base,
        cam_params=cam_init,
        window_media=media_truth,
        cam_to_window=cam2win_truth,
        window_planes=plane_biased,
    )

    cb_counter = {"k": 0}
    def progress_cb(phase, ray_rmse, len_rmse, cost):
        if not SHOW_PROGRESS:
            return
        cb_counter["k"] += 1
        nfev_est = cb_counter["k"] * 10
        print(
            f"[ExpB][bias={bias:+.2f}][nfev~{nfev_est:4d}] {phase} | "
            f"ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | cost={cost:.6e}"
        )

    cfg = RefractiveBAConfig(
        stage=0,
        verbosity=0,
        max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0),
    )

    opt = RefractiveBAOptimizer(
        dataset=dataset,
        cam_params=cam_init,
        cams_cpp=cams_cpp,
        cam_to_window=cam2win_truth,
        window_media=media_truth,
        window_planes=plane_biased,
        wand_length=WAND_LEN_MM,
        config=cfg,
        progress_callback=progress_cb,
    )
    opt._compute_physical_sigmas()

    # 实验B：相机+平面一起优化
    opt._optimize_generic(
        mode="expB_joint",
        description="ExpB Joint",
        enable_planes=True,
        enable_cam_t=True,
        enable_cam_r=True,
        limit_rot_rad=np.radians(20.0),
        limit_trans_mm=50.0,
        limit_plane_d_mm=50.0,
        limit_plane_angle_rad=np.radians(10.0),
        ftol=1e-7,
        loss="linear",
    )

    lam = 2.0 * max(1, len(cam_init))
    _, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
        opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
    )
    ray_rmse = np.sqrt(S_ray / max(1, N_ray))
    len_rmse = np.sqrt(S_len / max(1, N_len)) if N_len > 0 else np.nan

    cam_t_err_mean, cam_r_err_mean = mean_cam_err(opt.cam_params, cam_truth)

    # 平面“回正”量：沿真值法向的平均位置误差
    plane_d_errs = []
    for wid in sorted(plane_truth.keys()):
        n_gt = np.asarray(plane_truth[wid]["plane_n"], dtype=float)
        pt_gt = np.asarray(plane_truth[wid]["plane_pt"], dtype=float)
        pt_est = np.asarray(opt.window_planes[wid]["plane_pt"], dtype=float)
        plane_d_errs.append(abs(float(np.dot(n_gt, pt_est - pt_gt))))
    plane_d_err_mean = float(np.mean(plane_d_errs)) if plane_d_errs else np.nan

    records_B.append({
        "bias_mm": float(bias),
        "ray_rmse_mm": float(ray_rmse),
        "len_rmse_mm": float(len_rmse),
        "cam_t_err_mean_mm": cam_t_err_mean,
        "cam_r_err_mean_deg": cam_r_err_mean,
        "plane_d_err_mean_mm": plane_d_err_mean,
        "N_ray": int(N_ray),
        "N_len": int(N_len),
    })

dfB = pd.DataFrame(records_B).sort_values("bias_mm").reset_index(drop=True)
display(dfB)

plt.figure(figsize=(7,4))
plt.plot(dfB["bias_mm"], dfB["ray_rmse_mm"], marker="o", label="ExpB joint")
plt.xlabel("Plane bias along normal (mm)")
plt.ylabel("Ray RMSE (mm)")
plt.title("Experiment B: Plane Bias vs Ray RMSE (Camera + Plane optimize)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



=== [ExpB] Bias -15.00 mm ===

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
  [ExpB Joint] optimizing 30 parameters (10 blocks)...
    Global Fixed Weighting: lambda=8.0 (N_cams=4)
    Initial: S_ray=1522.64, S_len=10.87 (J0=0.9243)
[ExpB][bias=-15.00][nfev~  10] ExpB Joint | ray=0.796514 mm | len=0.190348 mm | cost=5.806694e+06
[ExpB][bias=-15.00][nfev~  20] ExpB Joint | ray=0.703952 mm | len=0.186725 mm | cost=4.535728e+06
[ExpB][bias=-15.00][nfev~  30] ExpB Joint | ray=0.520304 mm | len=0.179713 mm | cost=2.478249e+06
[ExpB][bias=-15.00][nfev~  40] ExpB Joint | ray=0.164755 mm | len=0.166543 mm | cost=2.492245e+05
[ExpB][bias=-15.00][nfev~  50] ExpB Joint | ray=0.038196 mm | len=0.143617 mm | cost=1.396943e+04
[ExpB][bias=-15.00][nfev~  60] ExpB Joint | ray=0.006807 mm | len=0.045448 mm | cost=4.859674e+02
[ExpB][bias=-15.00][nfev~  70] ExpB Joint | ray=0.003118 mm | len=0.010304 mm | cost=9.217069e+01
[ExpB][bias=-15.00][nfev~  80] ExpB Joint | 

,bias_mm,ray_rmse_mm,len_rmse_mm,cam_t_err_mean_mm,cam_r_err_mean_deg,plane_d_err_mean_mm,N_ray,N_len
0,-15.0,0.002181,0.010300,2.929833,1.086692,13.593156,2400,300
1,-12.5,0.002087,0.010521,2.316487,1.174601,10.808562,2400,300
2,-10.0,0.001335,0.005543,1.983529,1.153499,9.010493,2400,300
3,-7.5,0.001276,0.005090,1.412827,1.042935,6.429556,2400,300
4,-5.0,0.001079,0.002598,0.921168,1.061351,4.185523,2400,300
5,-2.5,0.001182,0.002806,0.371782,1.048978,1.765437,2400,300
6,0.0,0.001266,0.004915,0.251335,0.955202,0.433223,2400,300
7,2.5,0.000934,0.004314,0.579483,1.154927,1.180010,2400,300
8,5.0,0.000986,0.002292,1.231446,1.095595,1.881075,2400,300
9,7.5,0.010807,0.038555,2.528358,1.165975,3.165011,2400,300


In [3]:
import os, re, csv, glob, copy
import numpy as np
import cv2
import pandas as pd

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAOptimizer, RefractiveBAConfig
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import (
    CppCameraFactory
)

# -----------------------------
# Paths
# -----------------------------
init_cam_dir = r"H:\20260210\T0\Synthetic_test\camFile"   # 相机+平面初值都用这个
obs_csv      = r"H:\20260210\T0\synthetic_wand_points.csv"

# 可选：用于对比真值参数误差（不影响优化）
truth_cam_dir = r"H:\20260210\T0\Round1\camFile"

MAX_FRAMES = 0   # 先快跑；要全量改成 0
WAND_LEN_MM = 10.0

def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and not s.startswith("#"):
            return s, j
        j += 1
    return "", j

def parse_cam_file(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    d = {}
    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix") or s.startswith("# Camera Matrix K"):
            row, _ = _next_data(lines, i)
            d["f"] = float(row.split()[0])

        elif s.startswith("# Distortion"):
            row, _ = _next_data(lines, i)
            vals = [float(x) for x in row.split(",")]
            d["k1"] = vals[0] if len(vals) > 0 else 0.0
            d["k2"] = vals[1] if len(vals) > 1 else 0.0

        elif s.startswith("# Rotation Matrix R") or s.startswith("# Rotation Matrix:"):
            r1, _ = _next_data(lines, i)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([
                [float(x) for x in r1.split()],
                [float(x) for x in r2.split()],
                [float(x) for x in r3.split()],
            ], dtype=float)

        elif s.startswith("# Translation Vector T") or s.startswith("# Translation Vector:"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "WINDOW_ID=" in s:
            d["wid"] = int(s.split("WINDOW_ID=")[1])

        elif "plane.pt" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "plane.norm_vector" in s:
            row, _ = _next_data(lines, i)
            n = np.array([float(x) for x in row.split()], dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = float(row.split(",")[0])

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = [float(x) for x in row.split(",")]  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    d["cx"], d["cy"] = 640.0, 400.0
    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params, cam_to_window, window_planes, window_media = {}, {}, {}, {}

    for path in sorted(glob.glob(os.path.join(cam_dir, "cam*.txt"))):
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0)
        ], dtype=np.float64)
        cam_to_window[cid] = d["wid"]

        if with_plane and d["wid"] not in window_planes:
            # file是farthest；优化内部用closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            wid = d["wid"]
            window_planes[wid] = {
                "plane_pt": p_close,
                "plane_n": d["plane_n"],
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or pidx == 0:
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or pidx == 1:
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA,
        "obsB": obsB,
        "frames": frames,
        "cam_ids": None,
        "est_radius_small_mm": 1.5,
        "est_radius_large_mm": 2.0,
    }

def rot_err_deg(r_est, r_gt):
    R1, _ = cv2.Rodrigues(r_est)
    R2, _ = cv2.Rodrigues(r_gt)
    R = R2 @ R1.T
    c = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    return float(np.degrees(np.arccos(c)))

# -----------------------------
# Build init from Synthetic_test
# -----------------------------
cam_init, cam2win_init, plane_init, media_init = load_cam_set(init_cam_dir, with_plane=True)
dataset = load_obs(obs_csv, max_frames=MAX_FRAMES)
dataset["cam_ids"] = sorted(cam_init.keys())

class _Base: pass
base = _Base()
base.image_size = (800, 1280)

cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=media_init,
    cam_to_window=cam2win_init,
    window_planes=plane_init
)

# progress: 每10 nfev 打一次（BA内部每10次residual callback）
_cb = {"k": 0}
def progress_cb(phase, ray_rmse, len_rmse, cost):
    _cb["k"] += 1
    nfev_est = _cb["k"] * 10
    print(f"[nfev~{nfev_est:4d}] {phase} | ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | cost={cost:.6e}")

cfg = RefractiveBAConfig(stage=0, verbosity=0, max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0))
opt = RefractiveBAOptimizer(
    dataset=dataset,
    cam_params={k: v.copy() for k, v in cam_init.items()},
    cams_cpp=cams_cpp,
    cam_to_window=cam2win_init,
    window_media=copy.deepcopy(media_init),
    window_planes=copy.deepcopy(plane_init),
    wand_length=WAND_LEN_MM,
    config=cfg,
    progress_callback=progress_cb
)
opt._compute_physical_sigmas()

# --- 先评估初值 ---
lam = 2.0 * max(1, len(cam_init))
_, S_ray0, S_len0, N_ray0, N_len0 = opt.evaluate_residuals(opt.window_planes, opt.cam_params, lam, window_media=opt.window_media)
print("\nBefore optimize:")
print("ray_rmse_mm =", np.sqrt(S_ray0 / max(1, N_ray0)))
print("len_rmse_mm =", np.sqrt(S_len0 / max(1, N_len0)) if N_len0 > 0 else np.nan)

# --- ExpB: camera + plane optimize ---
opt._optimize_generic(
    mode="final",
    description="ExpB Joint From SyntheticInit",
    enable_planes=True,
    enable_cam_t=True,
    enable_cam_r=True,
    limit_rot_rad=np.radians(20.0),
    limit_trans_mm=50.0,
    limit_plane_d_mm=50.0,
    limit_plane_angle_rad=np.radians(10.0),
    ftol=1e-7,
    loss="linear",
)

# --- 优化后评估 ---
_, S_ray1, S_len1, N_ray1, N_len1 = opt.evaluate_residuals(opt.window_planes, opt.cam_params, lam, window_media=opt.window_media)
print("\nAfter optimize:")
print("ray_rmse_mm =", np.sqrt(S_ray1 / max(1, N_ray1)))
print("len_rmse_mm =", np.sqrt(S_len1 / max(1, N_len1)) if N_len1 > 0 else np.nan)
print("N_ray =", N_ray1, "N_len =", N_len1)

# --- 可选：对比Round1真值参数误差 ---
cam_truth, _, plane_truth, _ = load_cam_set(truth_cam_dir, with_plane=True)

t_errs, r_errs = [], []
for cid in sorted(cam_truth.keys()):
    p = opt.cam_params[cid]
    gt = cam_truth[cid]
    t_errs.append(np.linalg.norm(p[3:6] - gt[3:6]))
    r_errs.append(rot_err_deg(p[0:3], gt[0:3]))
print("\nVs Round1 truth (optional reference):")
print("cam_t_err_mean_mm =", float(np.mean(t_errs)))
print("cam_r_err_mean_deg =", float(np.mean(r_errs)))

plane_d_errs = []
for wid in sorted(plane_truth.keys()):
    n = np.asarray(plane_truth[wid]["plane_n"], float)
    pt_gt = np.asarray(plane_truth[wid]["plane_pt"], float)
    pt_est = np.asarray(opt.window_planes[wid]["plane_pt"], float)
    plane_d_errs.append(abs(float(np.dot(n, pt_est - pt_gt))))
print("plane_d_err_mean_mm =", float(np.mean(plane_d_errs)))



Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.

Before optimize:
ray_rmse_mm = 0.08848784389708853
len_rmse_mm = 0.256456298021347
  [ExpB Joint From SyntheticInit] optimizing 30 parameters (10 blocks)...
    [Constraint] S_ray/S_len (0.95) < 10. Forcing lambda=1.0 (was 8.0)
    Global Fixed Weighting: lambda=1.0 (N_cams=4)
    Initial: S_ray=156.60, S_len=164.42 (J0=0.0736)
[nfev~  10] ExpB Joint From SyntheticInit | ray=0.088488 mm | len=0.256456 mm | cost=6.995278e+05
[nfev~  20] ExpB Joint From SyntheticInit | ray=0.088488 mm | len=0.256456 mm | cost=6.995278e+05
[nfev~  30] ExpB Joint From SyntheticInit | ray=0.088488 mm | len=0.256456 mm | cost=6.995278e+05
[nfev~  40] ExpB Joint From SyntheticInit | ray=0.085074 mm | len=0.256956 mm | cost=6.244925e+05
[nfev~  50] ExpB Joint From SyntheticInit | ray=0.085074 mm | len=0.256956 mm | cost=6.244945e+05


KeyboardInterrupt: 

Debug on the final two rounds in BA, using the same input as Exp B.

In [8]:
import os, re, csv, glob, copy
import numpy as np
import cv2

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAOptimizer, RefractiveBAConfig
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import (
    CppCameraFactory
)

# =========================
# Paths (same as ExpB)
# =========================
init_cam_dir = r"H:\20260210\T0\Synthetic_test\camFile"
obs_csv      = r"H:\20260210\T0\synthetic_wand_points.csv"

MAX_FRAMES = 0  # 0 = all frames, set to 300 if you want quick run
WAND_LEN_MM = 10.0

# -------------------------
# Helpers (same loading logic as ExpB)
# -------------------------
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and not s.startswith("#"):
            return s, j
        j += 1
    return "", j

def parse_cam_file(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    d = {}
    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix") or s.startswith("# Camera Matrix K"):
            row, _ = _next_data(lines, i)
            d["f"] = float(row.split()[0])

        elif s.startswith("# Distortion"):
            row, _ = _next_data(lines, i)
            vals = [float(x) for x in row.split(",")]
            d["k1"] = vals[0] if len(vals) > 0 else 0.0
            d["k2"] = vals[1] if len(vals) > 1 else 0.0

        elif s.startswith("# Rotation Matrix R") or s.startswith("# Rotation Matrix:"):
            r1, _ = _next_data(lines, i)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([
                [float(x) for x in r1.split()],
                [float(x) for x in r2.split()],
                [float(x) for x in r3.split()],
            ], dtype=float)

        elif s.startswith("# Translation Vector T") or s.startswith("# Translation Vector:"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "WINDOW_ID=" in s:
            d["wid"] = int(s.split("WINDOW_ID=")[1])

        elif "plane.pt" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array([float(x) for x in row.split()], dtype=float)

        elif "plane.norm_vector" in s:
            row, _ = _next_data(lines, i)
            n = np.array([float(x) for x in row.split()], dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = float(row.split(",")[0])

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = [float(x) for x in row.split(",")]  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    d["cx"], d["cy"] = 640.0, 400.0
    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params, cam_to_window = {}, {}
    window_planes, window_media = {}, {}

    for path in sorted(glob.glob(os.path.join(cam_dir, "cam*.txt"))):
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0)
        ], dtype=np.float64)
        cam_to_window[cid] = d["wid"]

        if with_plane and d["wid"] not in window_planes:
            # file stores farthest interface; optimizer uses closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            wid = d["wid"]
            window_planes[wid] = {
                "plane_pt": p_close,
                "plane_n": d["plane_n"],
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA,
        "obsB": obsB,
        "frames": frames,
        "cam_ids": None,
        "est_radius_small_mm": 1.5,
        "est_radius_large_mm": 2.0,
    }

def eval_rmse(opt, tag):
    lam = 2.0 * max(1, len(opt.cam_params))
    _, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
        opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
    )
    ray_rmse = np.sqrt(S_ray / max(1, N_ray))
    len_rmse = np.sqrt(S_len / max(1, N_len)) if N_len > 0 else np.nan
    print(f"[{tag}] ray_rmse_mm={ray_rmse:.9f}, len_rmse_mm={len_rmse:.9f}, N_ray={N_ray}, N_len={N_len}")
    return ray_rmse, len_rmse, N_ray, N_len

# =========================
# Build EXACT ExpB state
# =========================
cam_init, cam_to_window, window_planes_init, window_media_init = load_cam_set(init_cam_dir, with_plane=True)
dataset = load_obs_dataset(obs_csv, max_frames=MAX_FRAMES)
dataset["cam_ids"] = sorted(cam_init.keys())

class _Base: pass
base = _Base()
base.image_size = (800, 1280)

cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=window_media_init,
    cam_to_window=cam_to_window,
    window_planes=window_planes_init
)

# optional progress print (nfev~10,20,...)
_cb = {"k": 0}
def progress_cb(phase, ray_rmse, len_rmse, cost):
    _cb["k"] += 1
    nfev_est = _cb["k"] * 10
    print(f"[nfev~{nfev_est:4d}] {phase} | ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | cost={cost:.6e}")

cfg = RefractiveBAConfig(
    stage=0,      # IMPORTANT: we manually run rounds; no optimize() loops
    verbosity=0,
    max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0),
)

opt = RefractiveBAOptimizer(
    dataset=dataset,
    cam_params={k: v.copy() for k, v in cam_init.items()},
    cams_cpp=cams_cpp,
    cam_to_window=cam_to_window,
    window_media=copy.deepcopy(window_media_init),
    window_planes=copy.deepcopy(window_planes_init),
    wand_length=WAND_LEN_MM,
    config=cfg,
    progress_callback=progress_cb
)
opt._compute_physical_sigmas()

# sanity snapshot hashes (confirm same state style as ExpB)
print("cam_ids:", sorted(opt.cam_params.keys()))
print("frames:", len(opt.fids_optim), "first10:", opt.fids_optim[:10])

import inspect
import modules.camera_calibration.wand_calibration.refraction_calibration_BA as ba_mod

print("module file:", ba_mod.__file__)
print("default losses:",
      ba_mod.RefractiveBAConfig().loss_plane,
      ba_mod.RefractiveBAConfig().loss_cam,
      ba_mod.RefractiveBAConfig().loss_joint,
      ba_mod.RefractiveBAConfig().loss_round4)

# 在你构造opt之后再看
print("opt losses:",
      opt.config.loss_plane, opt.config.loss_cam, opt.config.loss_joint, opt.config.loss_round4)

import numpy as np
import modules.camera_calibration.wand_calibration.refraction_calibration_BA as ba_mod

# def old_point_to_ray_dist(X, o, d):
#     d = d / (np.linalg.norm(d) + 1e-12)
#     v = X - o
#     t = float(np.dot(v, d))
#     if t >= 0.0:
#         v_perp = v - t * d
#         return float(np.linalg.norm(v_perp))
#     else:
#         return float(np.linalg.norm(v))

# # 关键：改 BA 模块内部实际调用的符号
# ba_mod.point_to_ray_dist = old_point_to_ray_dist


# 看调用点真实传参
print(inspect.getsource(ba_mod.RefractiveBAOptimizer.optimize).splitlines()[145:190])  # 可调整范围

# =========================
# 0) Before
# =========================
eval_rmse(opt, "before")

# =========================
# 1) Joint round only (BA joint settings)
# =========================
opt._optimize_generic(
    mode="joint",
    description="Notebook Joint",
    enable_planes=True,
    enable_cam_t=True,
    enable_cam_r=True,
    limit_rot_rad=np.radians(20.0),
    limit_trans_mm=50.0,
    limit_plane_d_mm=50.0,
    limit_plane_angle_rad=np.radians(10.0),
    ftol=1e-6,
    loss=opt.config.loss_joint,   # now config-controlled
)
eval_rmse(opt, "after_joint")

# =========================
# 2) Final refined round (BA final_refined settings)
# =========================
enable_k1 = opt.config.dist_coeff_num >= 1
enable_k2 = opt.config.dist_coeff_num >= 2

opt._optimize_generic(
    mode="final_refined",
    description="Notebook FinalRefined",
    enable_planes=True,
    enable_cam_t=True,
    enable_cam_r=True,
    enable_cam_f=True,
    enable_win_t=True,
    enable_cam_k1=enable_k1,
    enable_cam_k2=enable_k2,
    limit_rot_rad=np.radians(20.0),
    limit_trans_mm=50.0,
    limit_plane_d_mm=10.0,
    limit_plane_angle_rad=np.radians(5.0),
    ftol=1e-6,
    loss=opt.config.loss_round4,  # now config-controlled
)
eval_rmse(opt, "after_final_refined")



Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
cam_ids: [0, 1, 2, 3]
frames: 2500 first10: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
module file: D:\0.Code\OpenLPTGUI\OpenLPT\modules\camera_calibration\wand_calibration\refraction_calibration_BA.py
default losses: linear linear linear linear
opt losses: linear linear linear linear
[]
[before] ray_rmse_mm=0.232330749, len_rmse_mm=0.256456298, N_ray=20000, N_len=2500
  [Notebook Joint] optimizing 30 parameters (10 blocks)...
    [Constraint] S_ray/S_len (6.57) < 10. Forcing lambda=1.0 (was 8.0)
    Global Fixed Weighting: lambda=1.0 (N_cams=4)
    Initial: S_ray=1079.55, S_len=164.42 (J0=0.1197)
[nfev~  10] Notebook Joint | ray=0.232331 mm | len=0.256456 mm | cost=4.385429e+06
[nfev~  20] Notebook Joint | ray=0.232331 mm | len=0.256456 mm | cost=4.385429e+06
[nfev~  30] Notebook Joint | ray=0.232331 mm | len=0.256456 mm | cost=4.385429e+06
[nfev~  40] Notebook Joint | ray=0.207979 mm | len=0.256742 mm | cost=3.5040

(np.float64(0.00033583123883770783),
 np.float64(0.042863510927315066),
 20000,
 2500)

In [1]:
# =========================
# Single Cell: load -> align -> same joint optimize (robust parser)
# =========================
import os
import re
import csv
import glob
import copy
import numpy as np
import cv2

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAConfig, RefractiveBAOptimizer
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import CppCameraFactory
from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_rays_cpp_batch,
    triangulate_point,
    align_world_y_to_plane_intersection,
)

# ---------- user paths ----------
init_cam_dir = r"H:\20260210\T0\Synthetic_test\camFile_joint"
obs_csv      = r"H:\20260210\T0\synthetic_wand_points.csv"

# ---------- knobs ----------
MAX_FRAMES  = 0       # 0 = all frames
WAND_LEN_MM = 10.0
IMG_SIZE    = (800, 1280)


# ---------- helpers ----------
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and (not s.startswith("#")):
            return s, j
        j += 1
    return None, j

def _to_floats(s):
    # 支持 "1,2,3" / "1 2 3" / 混合分隔
    toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
    return [float(t) for t in toks]

def _to_ints(s):
    toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
    return [int(float(t)) for t in toks]  # 防 "800.0"

def parse_cam_file(path):
    import re
    import numpy as np
    import cv2

    def _next_data(lines, i):
        j = i + 1
        while j < len(lines):
            s = lines[j].strip()
            if s and (not s.startswith("#")):
                return s, j
            j += 1
        return None, j

    def _to_floats(s):
        toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
        return [float(t) for t in toks]

    def _to_ints(s):
        toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
        return [int(float(t)) for t in toks]

    d = {}
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Model"):
            row, _ = _next_data(lines, i)
            d["model_name"] = row.strip() if row is not None else "PINPLATE"

        elif s.startswith("# Image Size"):
            row, _ = _next_data(lines, i)
            rr = _to_ints(row)
            d["n_row"], d["n_col"] = rr[0], rr[1]

        elif s.startswith("# Camera Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            K = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)
            d["K"] = K
            d["f"] = float(K[0, 0])
            d["cx"] = float(K[0, 2])
            d["cy"] = float(K[1, 2])

        elif s.startswith("# Distortion Coefficients"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)
            d["k1"] = rr[0] if len(rr) > 0 else 0.0
            d["k2"] = rr[1] if len(rr) > 1 else 0.0

        elif s.startswith("# Rotation Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)

        elif s.startswith("# Translation Vector"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane reference point" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane normal" in s:
            row, _ = _next_data(lines, i)
            n = np.array(_to_floats(row), dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = _to_floats(row)[0]

        elif s.startswith("# WINDOW_ID="):
            d["wid_meta"] = int(s.split("=", 1)[1])

    # 关键：永远由 R 反算 rvec
    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()

    need = ["rvec", "tvec", "f", "cx", "cy", "plane_far", "plane_n", "n1", "n2", "n3", "thickness"]
    miss = [k for k in need if k not in d]
    if miss:
        raise ValueError(f"Missing keys in {path}: {miss}")

    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params, cam_to_window = {}, {}
    window_planes, window_media = {}, {}

    cam_files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    assert len(cam_files) > 0, f"No cam*.txt in {cam_dir}"

    for path in cam_files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0)
        ], dtype=np.float64)

        # 优先使用文件 meta 里的 WINDOW_ID
        if "wid_meta" in d:
            wid = int(d["wid_meta"])
        else:
            wid = 0 if cid <= 4 else 1  # fallback
        cam_to_window[cid] = wid

        if with_plane and (wid not in window_planes):
            # 文件存的是 farthest interface；优化器用 closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            window_planes[wid] = {
                "plane_pt": p_close,
                "plane_n": d["plane_n"],
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }

    return cam_params, cam_to_window, window_planes, window_media


def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA,
        "obsB": obsB,
        "frames": frames,
        "cam_ids": None,
        "est_radius_small_mm": 1.5,
        "est_radius_large_mm": 2.0,
    }


def eval_rmse(opt, tag):
    lam = 2.0 * max(1, len(opt.cam_params))
    _, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
        opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
    )
    ray_rmse = np.sqrt(S_ray / max(1, N_ray))
    len_rmse = np.sqrt(S_len / max(1, N_len)) if N_len > 0 else np.nan
    print(f"[{tag}] ray_rmse_mm={ray_rmse:.9f}, len_rmse_mm={len_rmse:.9f}, N_ray={N_ray}, N_len={N_len}")
    return ray_rmse, len_rmse, N_ray, N_len


def collect_points_for_alignment(opt):
    dataset = opt.dataset
    cams_cpp = opt.cams_cpp
    cam_params = opt.cam_params
    cam_to_window = opt.cam_to_window
    pts = []

    def _tri_all(obs_dict):
        out = []
        rays_by_fid = {fid: [] for fid in obs_dict.keys()}
        per_cam = {}

        for fid, cam_obs in obs_dict.items():
            for cid, uv in cam_obs.items():
                if cid not in cams_cpp or cid not in cam_params:
                    continue
                per_cam.setdefault(cid, []).append((fid, uv))

        for cid, items in per_cam.items():
            wid = cam_to_window.get(cid)
            uv_list = [uv for _, uv in items]
            meta_list = [
                {"cam_id": cid, "window_id": wid, "frame_id": fid, "endpoint": "?"}
                for fid, _ in items
            ]
            rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta_list)
            for (fid, _), r in zip(items, rays):
                if r.valid:
                    rays_by_fid[fid].append(r)

        for fid in obs_dict.keys():
            rays = rays_by_fid[fid]
            if len(rays) >= 2:
                X, _, valid, _ = triangulate_point(rays)
                if valid:
                    out.append(X)
        return out

    pts.extend(_tri_all(dataset.get("obsA", {})))
    pts.extend(_tri_all(dataset.get("obsB", {})))
    return pts


# =========================
# 1) Load
# =========================
cam_init, cam_to_window, window_planes_init, window_media_init = load_cam_set(init_cam_dir, with_plane=True)
dataset = load_obs_dataset(obs_csv, max_frames=MAX_FRAMES)
dataset["cam_ids"] = sorted(cam_init.keys())

print("cam_to_window:", cam_to_window)
for cid, p in cam_init.items():
    print(f"cam{cid}: rvec_norm_deg={np.linalg.norm(p[:3])*180/np.pi:.3f}, tvec={p[3:6]}")


class _Base:
    pass
base = _Base()
base.image_size = IMG_SIZE

cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=window_media_init,
    cam_to_window=cam_to_window,
    window_planes=window_planes_init
)

cfg = RefractiveBAConfig(
    stage=0,
    verbosity=1,
    max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0),
    diff_step_mode_joint="auto",
    diff_step_mode_final="auto",
)

_cb = {"k": 0}
def progress_cb(phase, ray_rmse, len_rmse, cost):
    _cb["k"] += 1
    nfev_est = _cb["k"] * 10
    print(f"[nfev~{nfev_est:4d}] {phase} | ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | cost={cost:.6e}")


opt = RefractiveBAOptimizer(
    dataset=dataset,
    cam_params={k: v.copy() for k, v in cam_init.items()},
    cams_cpp=cams_cpp,
    cam_to_window=cam_to_window,
    window_media=copy.deepcopy(window_media_init),
    window_planes=copy.deepcopy(window_planes_init),
    wand_length=WAND_LEN_MM,
    config=cfg,
    progress_callback=progress_cb
)
opt._compute_physical_sigmas()

print("cam_ids:", sorted(opt.cam_params.keys()))
print("window_ids:", sorted(opt.window_planes.keys()))
print("frames:", len(opt.fids_optim), "first10:", opt.fids_optim[:10])

# =========================
# 2) Before align
# =========================
eval_rmse(opt, "before_align")

# =========================
# 3) Align (rotate + shift)
# =========================
pts3d = collect_points_for_alignment(opt)
print(f"[align] points_for_centroid={len(pts3d)}")

ret = align_world_y_to_plane_intersection(
    opt.window_planes, opt.cam_params, points_3d=pts3d
)
if len(ret) == 5:
    cam_aligned, planes_aligned, _, R_align, t_shift = ret
else:
    cam_aligned, planes_aligned, _, R_align = ret
    t_shift = np.zeros(3, dtype=float)

opt.cam_params = {int(k): np.array(v, dtype=np.float64) for k, v in cam_aligned.items()}
opt.window_planes = copy.deepcopy(planes_aligned)
opt.sync_cpp_state(
    cam_params=opt.cam_params,
    window_planes=opt.window_planes,
    window_media=opt.window_media
)

# 用对齐后状态作为新初值
opt._sync_initial_state()
opt._compute_physical_sigmas()

print("[align] done")
print("[align] R_align=\n", R_align)
print("[align] t_shift=", t_shift)

eval_rmse(opt, "after_align_before_joint")

# =========================
# 4) Same JOINT optimize
# =========================
opt._optimize_generic(
    mode="joint",
    description="Notebook Joint",
    enable_planes=True,
    enable_cam_t=True,
    enable_cam_r=True,
    limit_rot_rad=np.radians(20.0),
    limit_trans_mm=50.0,
    limit_plane_d_mm=50.0,
    limit_plane_angle_rad=np.radians(10.0),
    ftol=1e-6,
    xtol=1e-6,
    gtol=1e-6,
    max_nfev=300,
    loss=opt.config.loss_joint,
)

eval_rmse(opt, "after_joint_from_aligned_init")
opt._print_plane_diagnostics("Joint End (Aligned Init)")


cam_to_window: {0: 0, 1: 1, 2: 1, 3: 0}
cam0: rvec_norm_deg=0.909, tvec=[-3.5592542 -1.4794667  6.6928316]
cam1: rvec_norm_deg=89.062, tvec=[184.65774    -7.7781103 201.6052   ]
cam2: rvec_norm_deg=91.307, tvec=[184.56527    -7.7120511 203.98762  ]
cam3: rvec_norm_deg=17.570, tvec=[-1.2466151 55.433966  16.507631 ]

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
Unit Normalization: sigma_ray=0.0224mm (1.0px at Z=201.4mm), sigma_wand=0.2000mm (2.0%)
cam_ids: [0, 1, 2, 3]
window_ids: [0, 1]
frames: 2500 first10: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[before_align] ray_rmse_mm=0.092704995, len_rmse_mm=0.243131943, N_ray=20000, N_len=2500
[align] points_for_centroid=5000
[Coordinate Alignment] Intersection line direction: [-0.2016, 0.9761, -0.0815]
[Coordinate Alignment] Intersection line direction: [-0.2016, 0.9761, -0.0815]
[ALIGN CHECK] Y-Axis (Intersection): [-4.27069348e-18  1.00000000e+00  1.89668606e-17]
[ALIGN CHECK] Z-Axis (Plane 0 Normal): [-2.01129160e

add location perturbation to plane and check calibration

In [1]:
# ============================================
# Realtime parallel test for plane shift impact
# ============================================

import os
import re
import csv
import glob
import copy
import time
import math
import threading
import traceback
import contextlib
import io

import numpy as np
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Markdown

from modules.camera_calibration.wand_calibration.refraction_calibration_BA import (
    RefractiveBAConfig, RefractiveBAOptimizer
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import CppCameraFactory

# -----------------------------
# Inputs
# -----------------------------
init_cam_dir = r"J:\Fish\R2\camFile"
obs_csv      = r"J:\Fish\R2\wand_points_filtered.csv"

SHIFTS_MM   = [-5.0, -2.0, 0.0, 2.0, 5.0]
MAX_WORKERS = 1 #len(SHIFTS_MM)   # 每个shift一个worker
MAX_FRAMES  = 0                # 0 = all
WAND_LEN_MM = 10.0
IMG_SIZE    = (800, 1280)

# 若你发现5并发卡住（C++对象并发初始化问题），临时改成2或3：
# MAX_WORKERS = 2


# -----------------------------
# Parse / Load helpers
# -----------------------------
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and (not s.startswith("#")):
            return s, j
        j += 1
    return None, j

def _to_floats(s):
    toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
    return [float(t) for t in toks]

def parse_cam_file(path):
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            K = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)
            d["f"]  = float(K[0, 0])
            d["cx"] = float(K[0, 2])
            d["cy"] = float(K[1, 2])

        elif s.startswith("# Distortion Coefficients"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)
            d["k1"] = rr[0] if len(rr) > 0 else 0.0
            d["k2"] = rr[1] if len(rr) > 1 else 0.0

        elif s.startswith("# Rotation Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)

        elif s.startswith("# Translation Vector"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane reference point" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane normal" in s:
            row, _ = _next_data(lines, i)
            n = np.array(_to_floats(row), dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = _to_floats(row)[0]

        elif s.startswith("# WINDOW_ID="):
            d["wid_meta"] = int(s.split("=", 1)[1])

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()

    need = ["rvec", "tvec", "f", "cx", "cy", "plane_far", "plane_n", "n1", "n2", "n3", "thickness"]
    miss = [k for k in need if k not in d]
    if miss:
        raise ValueError(f"Missing keys in {path}: {miss}")
    return d

def load_cam_set(cam_dir, with_plane=True):
    cam_params, cam_to_window = {}, {}
    window_planes, window_media = {}, {}

    cam_files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    if not cam_files:
        raise RuntimeError(f"No cam*.txt in {cam_dir}")

    for path in cam_files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0)
        ], dtype=np.float64)

        wid = int(d["wid_meta"]) if "wid_meta" in d else (0 if cid <= 4 else 1)
        cam_to_window[cid] = wid

        if with_plane and (wid not in window_planes):
            # file stores farthest interface; optimizer uses closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            window_planes[wid] = {
                "plane_pt": np.array(p_close, dtype=np.float64),
                "plane_n": np.array(d["plane_n"], dtype=np.float64),
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA,
        "obsB": obsB,
        "frames": frames,
        "cam_ids": None,
        "est_radius_small_mm": 1.5,
        "est_radius_large_mm": 2.0,
    }


# -----------------------------
# Optimization helpers
# -----------------------------
def eval_rmse(opt):
    lam = 2.0 * max(1, len(opt.cam_params))
    _, S_ray, S_len, N_ray, N_len = opt.evaluate_residuals(
        opt.window_planes, opt.cam_params, lam, window_media=opt.window_media
    )
    ray_rmse = math.sqrt(S_ray / max(1, N_ray))
    len_rmse = math.sqrt(S_len / max(1, N_len)) if N_len > 0 else float("nan")
    return ray_rmse, len_rmse, S_ray, S_len

def _set_status(status_map, lock, shift_mm, **kwargs):
    with lock:
        status_map[shift_mm].update(kwargs)

def build_optimizer(cam_params, cam_to_window, window_planes, window_media, dataset, progress_cb):
    class _Base: pass
    base = _Base()
    base.image_size = IMG_SIZE
    base.dist_coeff_num = 0

    cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
        base=base,
        cam_params=cam_params,
        window_media=window_media,
        cam_to_window=cam_to_window,
        window_planes=window_planes
    )

    cfg = RefractiveBAConfig(
        stage=0,
        verbosity=0,
        max_frames=(MAX_FRAMES if MAX_FRAMES > 0 else 0),
        diff_step_mode_joint="auto",
        diff_step_mode_final="auto",
    )

    opt = RefractiveBAOptimizer(
        dataset=dataset,
        cam_params={k: v.copy() for k, v in cam_params.items()},
        cams_cpp=cams_cpp,
        cam_to_window=cam_to_window,
        window_media=copy.deepcopy(window_media),
        window_planes=copy.deepcopy(window_planes),
        wand_length=WAND_LEN_MM,
        config=cfg,
        progress_callback=progress_cb
    )
    opt._compute_physical_sigmas()
    return opt

def run_one_shift(shift_mm, cam_init, cam_to_window, window_planes_init, window_media_init, dataset, status_map, lock):
    try:
        _set_status(status_map, lock, shift_mm, status="starting", nfev_est=0, ray_rmse=None, len_rmse=None, error="", t_last=time.time())

        cam_params = {k: v.copy() for k, v in cam_init.items()}
        window_planes = copy.deepcopy(window_planes_init)
        window_media = copy.deepcopy(window_media_init)

        _set_status(status_map, lock, shift_mm, status="perturbing", t_last=time.time())
        for wid, pl in window_planes.items():
            n = np.asarray(pl["plane_n"], dtype=float)
            n = n / (np.linalg.norm(n) + 1e-12)
            pt = np.asarray(pl["plane_pt"], dtype=float)
            pl["plane_n"] = n
            pl["plane_pt"] = pt + shift_mm * n

        nfev_counter = {"k": 0}
        def progress_cb(phase, ray_rmse, len_rmse, cost):
            nfev_counter["k"] += 1
            _set_status(
                status_map, lock, shift_mm,
                status="optimizing",
                nfev_est=nfev_counter["k"] * 10,
                ray_rmse=float(ray_rmse) if ray_rmse is not None else None,
                len_rmse=float(len_rmse) if len_rmse is not None else None,
                cost=float(cost) if cost is not None else None,
                t_last=time.time()
            )

        t0 = time.time()
        _set_status(status_map, lock, shift_mm, status="building_cpp", t_last=time.time())

        # 为了不刷屏，静默内部print
        with contextlib.redirect_stdout(io.StringIO()):
            opt = build_optimizer(cam_params, cam_to_window, window_planes, window_media, dataset, progress_cb)

        _set_status(status_map, lock, shift_mm, status="eval_before", t_last=time.time())
        with contextlib.redirect_stdout(io.StringIO()):
            ray0, len0, Sray0, Slen0 = eval_rmse(opt)

        _set_status(status_map, lock, shift_mm, status="optimizing", t_last=time.time())
        with contextlib.redirect_stdout(io.StringIO()):
            res, _ = opt._optimize_generic(
                mode="joint",
                description=f"Joint shift {shift_mm:+.1f}mm",
                enable_planes=True,
                enable_cam_t=True,
                enable_cam_r=True,
                limit_rot_rad=np.radians(20.0),
                limit_trans_mm=50.0,
                limit_plane_d_mm=50.0,
                limit_plane_angle_rad=np.radians(10.0),
                ftol=1e-6,
                xtol=1e-6,
                gtol=1e-6,
                loss=opt.config.loss_joint,
                max_nfev=20,
            )

        _set_status(status_map, lock, shift_mm, status="eval_after", t_last=time.time())
        with contextlib.redirect_stdout(io.StringIO()):
            ray1, len1, Sray1, Slen1 = eval_rmse(opt)

        elapsed = time.time() - t0
        nfev = getattr(res, "nfev", None)

        out = {
            "shift_mm": shift_mm,
            "status": "done",
            "elapsed_s": elapsed,
            "nfev": int(nfev) if nfev is not None else None,
            "ray_before": float(ray0),
            "len_before": float(len0),
            "ray_after": float(ray1),
            "len_after": float(len1),
            "Sray_before": float(Sray0),
            "Slen_before": float(Slen0),
            "Sray_after": float(Sray1),
            "Slen_after": float(Slen1),
            "t_last": time.time(),
        }
        _set_status(status_map, lock, shift_mm, **out)
        return out

    except Exception as e:
        _set_status(
            status_map, lock, shift_mm,
            status="failed",
            error=f"{type(e).__name__}: {e}",
            traceback=traceback.format_exc(),
            t_last=time.time(),
        )
        raise


# -----------------------------
# Main runner
# -----------------------------
cam_init, cam_to_window, window_planes_init, window_media_init = load_cam_set(init_cam_dir, with_plane=True)
dataset = load_obs_dataset(obs_csv, max_frames=MAX_FRAMES)
dataset["cam_ids"] = sorted(cam_init.keys())

status_map = {}
lock = threading.Lock()
for s in SHIFTS_MM:
    status_map[s] = {
        "status": "queued",
        "nfev_est": 0,
        "ray_rmse": None,
        "len_rmse": None,
        "cost": None,
        "error": "",
        "t_last": time.time(),
    }

futures = []
results = []
start_all = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for s in SHIFTS_MM:
        futures.append(ex.submit(
            run_one_shift, s,
            cam_init, cam_to_window, window_planes_init, window_media_init, dataset,
            status_map, lock
        ))

    panel = display(Markdown("Starting workers..."), display_id=True)

    while True:
        done_cnt = sum(1 for f in futures if f.done())
        now = time.time()

        lines = []
        lines.append("### Plane Shift Joint Test (Realtime)")
        lines.append(f"- Done: **{done_cnt}/{len(futures)}**")
        lines.append(f"- Elapsed: **{now - start_all:.1f} s**")
        lines.append("")
        lines.append("| Shift (mm) | Status | nfev~ | Ray RMSE (mm) | Len RMSE (mm) | Last update (s ago) | Error |")
        lines.append("|---:|:---|---:|---:|---:|---:|:---|")

        with lock:
            for s in sorted(SHIFTS_MM):
                st = status_map[s]
                ray = st.get("ray_rmse")
                ln = st.get("len_rmse")
                err = st.get("error", "")
                if len(err) > 40:
                    err = err[:40] + "..."
                lag = now - st.get("t_last", now)

                ray_txt = "-" if ray is None else f"{ray:.6f}"
                len_txt = "-" if ln is None else f"{ln:.6f}"

                lines.append(
                    f"| {s:+.1f} | {st.get('status','?')} | {st.get('nfev_est',0)} | "
                    f"{ray_txt} | {len_txt} | {lag:.1f} | {err} |"
                )

        panel.update(Markdown("\n".join(lines)))

        if done_cnt == len(futures):
            break
        time.sleep(0.4)

    for f in as_completed(futures):
        try:
            results.append(f.result())
        except Exception:
            pass

# Final summary
ok = [r for r in results if r.get("status") == "done"]
fail = [s for s in SHIFTS_MM if status_map[s].get("status") == "failed"]

lines = []
lines.append("### Plane Shift Joint Test (Completed)")
if ok:
    ok = sorted(ok, key=lambda x: x["ray_after"])
    lines.append("")
    lines.append("| Shift | nfev | Time(s) | Ray before | Ray after | Len after |")
    lines.append("|---:|---:|---:|---:|---:|---:|")
    for r in ok:
        lines.append(
            f"| {r['shift_mm']:+.1f} | {r['nfev']} | {r['elapsed_s']:.1f} | "
            f"{r['ray_before']:.6f} | {r['ray_after']:.6f} | {r['len_after']:.6f} |"
        )

if fail:
    lines.append("")
    lines.append("**Failed shifts**")
    for s in fail:
        lines.append(f"- {s:+.1f}: {status_map[s].get('error')}")

display(Markdown("\n".join(lines)))


### Plane Shift Joint Test (Realtime)
- Done: **5/5**
- Elapsed: **8093.1 s**

| Shift (mm) | Status | nfev~ | Ray RMSE (mm) | Len RMSE (mm) | Last update (s ago) | Error |
|---:|:---|---:|---:|---:|---:|:---|
| -5.0 | failed | 740 | 1.671083 | 0.676840 | 7030.7 | TypeError: _set_status() got multiple va... |
| -2.0 | failed | 630 | 1.616039 | 0.799694 | 5658.3 | TypeError: _set_status() got multiple va... |
| +0.0 | failed | 660 | 1.642280 | 0.724591 | 4300.4 | TypeError: _set_status() got multiple va... |
| +2.0 | failed | 740 | 1.674160 | 0.668414 | 2160.9 | TypeError: _set_status() got multiple va... |
| +5.0 | failed | 740 | 1.693519 | 0.610845 | 0.1 | TypeError: _set_status() got multiple va... |

### Plane Shift Joint Test (Completed)

**Failed shifts**
- -5.0: TypeError: _set_status() got multiple values for argument 'shift_mm'
- -2.0: TypeError: _set_status() got multiple values for argument 'shift_mm'
- +0.0: TypeError: _set_status() got multiple values for argument 'shift_mm'
- +2.0: TypeError: _set_status() got multiple values for argument 'shift_mm'
- +5.0: TypeError: _set_status() got multiple values for argument 'shift_mm'

Optimize both camera and points, using SBA

In [7]:
import os
import re
import csv
import glob
import copy
import time
import numpy as np
import cv2
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix

from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_rays_cpp_batch, point_to_ray_dist, update_normal_tangent, camera_center
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import CppCameraFactory, CppSyncAdapter


# =========================
# Inputs
# =========================
init_cam_dir = r"H:\20260210\T0\Synthetic_test\camFile_joint"
obs_csv      = r"H:\20260210\T0\synthetic_wand_points.csv"

MAX_FRAMES   = 0           # 0=all
WAND_LEN_MM  = 10.0
IMG_SIZE     = (800, 1280)

# 优化参数
MAX_NFEV     = 100
LOSS         = "linear"
FTOL = 1e-7
XTOL = None
GTOL = 1e-7


# =========================
# Data loaders
# =========================
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and (not s.startswith("#")):
            return s, j
        j += 1
    return None, j

def _to_floats(s):
    toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
    return [float(t) for t in toks]

def parse_cam_file(path):
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            K = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)
            d["f"] = float(K[0,0]); d["cx"] = float(K[0,2]); d["cy"] = float(K[1,2])

        elif s.startswith("# Distortion Coefficients"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)
            d["k1"] = rr[0] if len(rr)>0 else 0.0
            d["k2"] = rr[1] if len(rr)>1 else 0.0

        elif s.startswith("# Rotation Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)

        elif s.startswith("# Translation Vector"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane reference point" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane normal" in s:
            row, _ = _next_data(lines, i)
            n = np.array(_to_floats(row), dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)  # obj,win,air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = _to_floats(row)[0]

        elif s.startswith("# WINDOW_ID="):
            d["wid_meta"] = int(s.split("=", 1)[1])

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    return d

def load_cam_set(cam_dir):
    cam_params, cam_to_window = {}, {}
    window_planes, window_media = {}, {}

    cam_files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    if not cam_files:
        raise RuntimeError(f"No cam*.txt in {cam_dir}")

    for path in cam_files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1",0.0), d.get("k2",0.0)
        ], dtype=np.float64)

        wid = int(d["wid_meta"]) if "wid_meta" in d else (0 if cid <= 4 else 1)
        cam_to_window[cid] = wid

        if wid not in window_planes:
            # file stores farthest interface; optimizer uses closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            window_planes[wid] = {
                "plane_pt": np.array(p_close, dtype=np.float64),
                "plane_n": np.array(d["plane_n"], dtype=np.float64),
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }
    return cam_params, cam_to_window, window_planes, window_media

def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st  = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {
        "obsA": obsA, "obsB": obsB, "frames": frames
    }


# =========================
# Build initial state
# =========================
cam_init, cam_to_window, window_planes_init, window_media = load_cam_set(init_cam_dir)
dataset = load_obs_dataset(obs_csv, MAX_FRAMES)

cam_ids = sorted(cam_init.keys())
win_ids = sorted(window_planes_init.keys())
fids = [fid for fid in dataset["frames"] if (fid in dataset["obsA"] and fid in dataset["obsB"])]
print(f"cams={len(cam_ids)}, windows={len(win_ids)}, frames={len(fids)}")

class _Base: pass
base = _Base()
base.image_size = IMG_SIZE
base.dist_coeff_num = 0

cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=window_media,
    cam_to_window=cam_to_window,
    window_planes=window_planes_init
)

# 固定一个参考相机，去除全局 gauge
ref_cam = cam_ids[0]
opt_cam_ids = [c for c in cam_ids if c != ref_cam]
print("reference cam fixed:", ref_cam, "optimized cams:", opt_cam_ids)

# 预构建观测列表
obs_rows = []
for fid in fids:
    for cid, uv in dataset["obsA"].get(fid, {}).items():
        if cid in cam_ids:
            obs_rows.append((fid, cid, 0, uv))  # endpoint A=0
    for cid, uv in dataset["obsB"].get(fid, {}).items():
        if cid in cam_ids:
            obs_rows.append((fid, cid, 1, uv))  # endpoint B=1

# 点索引（每帧 A/B）
fid_to_idx = {fid:i for i,fid in enumerate(fids)}
nF = len(fids)


# =========================
# Initial 3D points by triangulation (one-time init)
# =========================
def triangulate_init_points():
    XA = np.zeros((nF,3), dtype=float)
    XB = np.zeros((nF,3), dtype=float)
    validA = np.zeros(nF, dtype=bool)
    validB = np.zeros(nF, dtype=bool)

    def tri_endpoint(obs_key, endpoint_flag, Xout, valid):
        per_cam = {}
        for i, fid in enumerate(fids):
            obs_f = dataset[obs_key].get(fid, {})
            for cid, uv in obs_f.items():
                if cid in cams_cpp:
                    per_cam.setdefault(cid, []).append((i, fid, uv))

        rays_by_frame = {i: [] for i in range(nF)}
        for cid, items in per_cam.items():
            wid = cam_to_window[cid]
            uv_list = [uv for (_,_,uv) in items]
            meta = [{"cam_id":cid,"window_id":wid,"frame_id":fid,"endpoint":"A" if endpoint_flag==0 else "B"} for (_,fid,_) in items]
            rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta)
            for (i,_,_), r in zip(items, rays):
                if r.valid:
                    rays_by_frame[i].append(r)

        for i in range(nF):
            rays = rays_by_frame[i]
            if len(rays) >= 2:
                # simple line-line LS triangulation
                # minimize sum ||(I-dd^T)(X-o)||^2
                A = np.zeros((3,3)); b = np.zeros(3)
                for r in rays:
                    d = r.d / (np.linalg.norm(r.d) + 1e-12)
                    o = r.o
                    P = np.eye(3) - np.outer(d,d)
                    A += P
                    b += P @ o
                try:
                    X = np.linalg.solve(A + 1e-9*np.eye(3), b)
                    Xout[i,:] = X
                    valid[i] = True
                except np.linalg.LinAlgError:
                    pass

    tri_endpoint("obsA", 0, XA, validA)
    tri_endpoint("obsB", 1, XB, validB)

    # fallback: invalid点用零或A/B互补
    for i in range(nF):
        if not validA[i] and validB[i]:
            XA[i] = XB[i] + np.array([0,0,-WAND_LEN_MM])
            validA[i] = True
        if not validB[i] and validA[i]:
            XB[i] = XA[i] + np.array([0,0, WAND_LEN_MM])
            validB[i] = True
        if not validA[i] and not validB[i]:
            XA[i] = np.array([0,0,0.0])
            XB[i] = np.array([0,0,WAND_LEN_MM])
            validA[i] = validB[i] = True

    return XA, XB

XA0, XB0 = triangulate_init_points()


# =========================
# Parameter packing
# =========================
cam_dim = 6 * len(opt_cam_ids)
# 每window: plane_pt(3) + tangent-normal delta (a,b)=2
win_dim = 5 * len(win_ids)
pts_dim = 6 * nF
n_params = cam_dim + win_dim + pts_dim

cam_off = 0
win_off = cam_off + cam_dim
pts_off = win_off + win_dim

# 保存固定相机内参
cam_fixed_intr = {cid: cam_init[cid][6:11].copy() for cid in cam_ids}
cam_r0 = {cid: cam_init[cid][0:3].copy() for cid in cam_ids}
cam_t0 = {cid: cam_init[cid][3:6].copy() for cid in cam_ids}

# 平面初值
pl_pt0 = {wid: window_planes_init[wid]["plane_pt"].copy() for wid in win_ids}
pl_n0  = {wid: window_planes_init[wid]["plane_n"].copy()  for wid in win_ids}

# 每窗口 anchor（用于 d-shift 更稳定）
pl_anchor = {}
for wid in win_ids:
    centers = []
    for cid, w in cam_to_window.items():
        if w == wid:
            p = cam_init[cid]
            R,_ = cv2.Rodrigues(p[:3]); C = camera_center(R, p[3:6])
            centers.append(C)
    if centers:
        pl_anchor[wid] = np.mean(np.asarray(centers), axis=0)
    else:
        pl_anchor[wid] = pl_pt0[wid].copy()

def pack_x0():
    x = np.zeros(n_params, dtype=float)

    # cams (excluding reference)
    k = cam_off
    for cid in opt_cam_ids:
        x[k:k+3] = cam_r0[cid]
        x[k+3:k+6] = cam_t0[cid]
        k += 6

    # planes
    k = win_off
    for wid in win_ids:
        # optimize plane_pt directly + (a,b)=0 init
        x[k:k+3] = pl_pt0[wid]
        x[k+3] = 0.0
        x[k+4] = 0.0
        k += 5

    # points
    k = pts_off
    for i in range(nF):
        x[k:k+3] = XA0[i]
        x[k+3:k+6] = XB0[i]
        k += 6

    return x

def unpack_params(x):
    # cams
    cam_params = {}
    for cid in cam_ids:
        p = cam_init[cid].copy()
        if cid == ref_cam:
            # reference cam fixed
            p[:3] = cam_r0[cid]
            p[3:6] = cam_t0[cid]
        else:
            j = opt_cam_ids.index(cid)
            base = cam_off + 6*j
            p[:3] = x[base:base+3]
            p[3:6] = x[base+3:base+6]
        p[6:11] = cam_fixed_intr[cid]
        cam_params[cid] = p

    # planes
    planes = {}
    for j, wid in enumerate(win_ids):
        base = win_off + 5*j
        pt = x[base:base+3]
        a = x[base+3]
        b = x[base+4]
        n = update_normal_tangent(pl_n0[wid], a, b)
        planes[wid] = {
            "plane_pt": pt,
            "plane_n": n,
            "initialized": True
        }

    # points
    XA = np.zeros((nF,3), dtype=float)
    XB = np.zeros((nF,3), dtype=float)
    for i in range(nF):
        base = pts_off + 6*i
        XA[i,:] = x[base:base+3]
        XB[i,:] = x[base+3:base+6]

    return cam_params, planes, XA, XB


# =========================
# Jacobian sparsity (SBA style)
# =========================
# residuals: one per observation + one per frame wand length
n_obs = len(obs_rows)
m = n_obs + nF
n = n_params

Jsp = lil_matrix((m, n), dtype=np.int8)

# obs residual: depends on camera(6, if not reference) + window plane(5) + point(3)
for r_idx, (fid, cid, endpoint, uv) in enumerate(obs_rows):
    # camera vars
    if cid != ref_cam:
        j = opt_cam_ids.index(cid)
        cbase = cam_off + 6*j
        Jsp[r_idx, cbase:cbase+6] = 1

    # window vars
    wid = cam_to_window[cid]
    wj = win_ids.index(wid)
    wbase = win_off + 5*wj
    Jsp[r_idx, wbase:wbase+5] = 1

    # point vars
    i = fid_to_idx[fid]
    pbase = pts_off + 6*i
    if endpoint == 0:
        Jsp[r_idx, pbase:pbase+3] = 1
    else:
        Jsp[r_idx, pbase+3:pbase+6] = 1

# wand residual: depends on XA/XB of this frame
for i, fid in enumerate(fids):
    r_idx = n_obs + i
    pbase = pts_off + 6*i
    Jsp[r_idx, pbase:pbase+6] = 1

Jsp = Jsp.tocsr()
print("jac_sparsity shape:", Jsp.shape, "nnz:", Jsp.nnz)


# =========================
# Residual function
# =========================
# 归一化
# 你当前配置改为1px；这里按几何估计sigma_ray
avg_f = np.mean([cam_init[c][6] for c in cam_ids])
# 用初始相机中心到原点的大致距离估计尺度
z_scale = []
for c in cam_ids:
    p = cam_init[c]
    R,_ = cv2.Rodrigues(p[:3]); C = camera_center(R, p[3:6])
    z_scale.append(abs(C[2]) + 1e-6)
avg_z = np.mean(z_scale)
sigma_ray = 1.0 * (avg_z / avg_f)  # 1px -> mm
sigma_wand = 0.02 * WAND_LEN_MM
print(f"sigma_ray={sigma_ray:.6f} mm, sigma_wand={sigma_wand:.6f} mm")

call_counter = {"k": 0, "t0": time.time()}

def residual_fun(x):
    cam_params, planes, XA, XB = unpack_params(x)

    # sync C++ camera objects with current cam+plane
    for cid in cam_ids:
        kw = CppSyncAdapter.build_update_kwargs(
            cam_params=cam_params,
            window_planes=planes,
            window_media=window_media,
            cam_to_window=cam_to_window,
            cam_id=cid
        )
        CppSyncAdapter.apply(cams_cpp, cid, kw)

    # batch build rays per camera for A/B
    per_cam_A = {}
    per_cam_B = {}
    for (fid, cid, endpoint, uv) in obs_rows:
        if endpoint == 0:
            per_cam_A.setdefault(cid, []).append((fid, uv))
        else:
            per_cam_B.setdefault(cid, []).append((fid, uv))

    ray_lookup_A = {}
    for cid, items in per_cam_A.items():
        wid = cam_to_window[cid]
        uv_list = [uv for (_,uv) in items]
        meta = [{"cam_id":cid, "window_id":wid, "frame_id":fid, "endpoint":"A"} for (fid,_) in items]
        rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta)
        for (fid,_), r in zip(items, rays):
            ray_lookup_A[(fid,cid)] = r

    ray_lookup_B = {}
    for cid, items in per_cam_B.items():
        wid = cam_to_window[cid]
        uv_list = [uv for (_,uv) in items]
        meta = [{"cam_id":cid, "window_id":wid, "frame_id":fid, "endpoint":"B"} for (fid,_) in items]
        rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta)
        for (fid,_), r in zip(items, rays):
            ray_lookup_B[(fid,cid)] = r

    res = np.zeros(m, dtype=float)

    S_ray = 0.0
    N_ray = 0

    # observation residuals
    for ridx, (fid, cid, endpoint, uv) in enumerate(obs_rows):
        i = fid_to_idx[fid]
        X = XA[i] if endpoint == 0 else XB[i]
        r = ray_lookup_A.get((fid,cid)) if endpoint == 0 else ray_lookup_B.get((fid,cid))
        if (r is not None) and r.valid:
            d = point_to_ray_dist(X, r.o, r.d)
            res[ridx] = d / sigma_ray
            S_ray += d*d
            N_ray += 1
        else:
            # invalid ray penalty
            res[ridx] = 100.0 / sigma_ray

    # wand length residuals
    S_len = 0.0
    for i, fid in enumerate(fids):
        L = np.linalg.norm(XB[i] - XA[i])
        dl = L - WAND_LEN_MM
        res[n_obs + i] = dl / sigma_wand
        S_len += dl*dl

    call_counter["k"] += 1
    if call_counter["k"] % 10 == 0:
        ray_rmse = np.sqrt(S_ray / max(1, N_ray))
        len_rmse = np.sqrt(S_len / max(1, nF))
        elapsed = time.time() - call_counter["t0"]
        print(f"[fve~{call_counter['k']:4d}] ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | elapsed={elapsed:.1f}s")

    return res


# =========================
# Run SBA
# =========================
x0 = pack_x0()

print("Start SBA...")
t_start = time.time()
res = least_squares(
    residual_fun,
    x0,
    method="trf",
    jac_sparsity=Jsp,
    loss=LOSS,
    ftol=FTOL,
    xtol=XTOL,
    gtol=GTOL,
    max_nfev=MAX_NFEV,
    verbose=0,
)
t_used = time.time() - t_start

print("\n=== SBA Done ===")
print("status:", res.status)
print("message:", res.message)
print("nfev:", res.nfev)
print("time(s):", round(t_used, 2))

cam_opt, plane_opt, XA_opt, XB_opt = unpack_params(res.x)

# quick post metrics
r_final = residual_fun(res.x)
cost = 0.5 * np.sum(r_final**2)
print("final cost:", cost)

# 你如果想看最终ray/len，可从 residual_fun 中再提一次，或者在这里复算一版统计


cams=4, windows=2, frames=2500

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
reference cam fixed: 0 optimized cams: [1, 2, 3]
jac_sparsity shape: (22500, 15028) nnz: 265000
sigma_ray=0.010261 mm, sigma_wand=0.200000 mm
Start SBA...
[fve~  10] ray=0.092705 mm | len=0.243132 mm | elapsed=5.0s
[fve~  20] ray=0.077318 mm | len=0.266043 mm | elapsed=10.0s
[fve~  30] ray=0.077318 mm | len=0.266043 mm | elapsed=15.0s
[fve~  40] ray=0.057624 mm | len=0.272857 mm | elapsed=20.0s
[fve~  50] ray=0.057624 mm | len=0.272857 mm | elapsed=25.0s
[fve~  60] ray=0.053122 mm | len=0.316978 mm | elapsed=30.0s
[fve~  70] ray=0.053121 mm | len=0.316978 mm | elapsed=34.9s
[fve~  80] ray=0.034577 mm | len=0.313114 mm | elapsed=39.9s
[fve~  90] ray=0.034577 mm | len=0.313114 mm | elapsed=44.9s
[fve~ 100] ray=0.029376 mm | len=0.314735 mm | elapsed=49.8s
[fve~ 110] ray=0.025007 mm | len=0.308888 mm | elapsed=54.8s
[fve~ 120] ray=0.025007 mm | len=0.308888 mm | elapsed=59.6s
[

KeyboardInterrupt: 

In [8]:
import os
import re
import csv
import glob
import time
import numpy as np
import cv2

from scipy.optimize import least_squares
from scipy.sparse import lil_matrix

from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_rays_cpp_batch,
    point_to_ray_dist,
    update_normal_tangent,
    camera_center,
    align_world_y_to_plane_intersection,
)
from modules.camera_calibration.wand_calibration.refraction_wand_calibrator import (
    CppCameraFactory, CppSyncAdapter
)

# =========================
# Inputs
# =========================
init_cam_dir = r"H:\20260210\T0\Synthetic_test\camFile_joint"
obs_csv      = r"H:\20260210\T0\synthetic_wand_points.csv"

MAX_FRAMES  = 0          # 0=all
WAND_LEN_MM = 10.0
IMG_SIZE    = (800, 1280)

MAX_NFEV = 150
FTOL = 1e-7
XTOL = None     # 你要求避免 xtol 早退
GTOL = 1e-6
LOSS = "linear"


# =========================
# Loaders
# =========================
def _next_data(lines, i):
    j = i + 1
    while j < len(lines):
        s = lines[j].strip()
        if s and (not s.startswith("#")):
            return s, j
        j += 1
    return None, j

def _to_floats(s):
    toks = [t for t in re.split(r"[\s,]+", s.strip()) if t]
    return [float(t) for t in toks]

def parse_cam_file(path):
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        s = line.strip()

        if s.startswith("# Camera Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            K = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)
            d["f"] = float(K[0, 0]); d["cx"] = float(K[0, 2]); d["cy"] = float(K[1, 2])

        elif s.startswith("# Distortion Coefficients"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)
            d["k1"] = rr[0] if len(rr) > 0 else 0.0
            d["k2"] = rr[1] if len(rr) > 1 else 0.0

        elif s.startswith("# Rotation Matrix"):
            r1, _ = _next_data(lines, i + 0)
            r2, _ = _next_data(lines, i + 1)
            r3, _ = _next_data(lines, i + 2)
            d["R"] = np.array([_to_floats(r1), _to_floats(r2), _to_floats(r3)], dtype=float)

        elif s.startswith("# Translation Vector"):
            row, _ = _next_data(lines, i)
            d["tvec"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane reference point" in s:
            row, _ = _next_data(lines, i)
            d["plane_far"] = np.array(_to_floats(row), dtype=float)

        elif "Refractive plane normal" in s:
            row, _ = _next_data(lines, i)
            n = np.array(_to_floats(row), dtype=float)
            d["plane_n"] = n / (np.linalg.norm(n) + 1e-12)

        elif s.startswith("# n_plate"):
            row, _ = _next_data(lines, i)
            rr = _to_floats(row)  # object, glass, air
            d["n3"], d["n2"], d["n1"] = rr[0], rr[1], rr[2]

        elif s.startswith("# w_array"):
            row, _ = _next_data(lines, i)
            d["thickness"] = _to_floats(row)[0]

        elif s.startswith("# WINDOW_ID="):
            d["wid_meta"] = int(s.split("=", 1)[1])

    rvec, _ = cv2.Rodrigues(d["R"])
    d["rvec"] = rvec.flatten()
    return d

def load_cam_set(cam_dir):
    cam_params, cam_to_window = {}, {}
    window_planes, window_media = {}, {}

    cam_files = sorted(glob.glob(os.path.join(cam_dir, "cam*.txt")))
    if not cam_files:
        raise RuntimeError(f"No cam*.txt found in {cam_dir}")

    for path in cam_files:
        cid = int(re.findall(r"cam(\d+)", os.path.basename(path))[0])
        d = parse_cam_file(path)

        cam_params[cid] = np.array([
            d["rvec"][0], d["rvec"][1], d["rvec"][2],
            d["tvec"][0], d["tvec"][1], d["tvec"][2],
            d["f"], d["cx"], d["cy"], d.get("k1", 0.0), d.get("k2", 0.0)
        ], dtype=np.float64)

        wid = int(d["wid_meta"]) if "wid_meta" in d else (0 if cid <= 4 else 1)
        cam_to_window[cid] = wid

        if wid not in window_planes:
            # file stores farthest; optimizer uses closest
            p_close = d["plane_far"] - d["plane_n"] * d["thickness"]
            window_planes[wid] = {
                "plane_pt": np.array(p_close, dtype=np.float64),
                "plane_n": np.array(d["plane_n"], dtype=np.float64),
                "initialized": True
            }
            window_media[wid] = {
                "n1": d["n1"], "n2": d["n2"], "n3": d["n3"],
                "thickness": d["thickness"]
            }

    return cam_params, cam_to_window, window_planes, window_media

def load_obs_dataset(csv_path, max_frames=0):
    obsA, obsB, frames = {}, {}, set()
    with open(csv_path, "r", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        for r in rd:
            fid = int(r["Frame"])
            cid = int(r["Camera"])
            st = r["Status"]
            pidx = int(r["PointIdx"])
            uv = np.array([float(r["X"]), float(r["Y"])], dtype=float)

            frames.add(fid)
            if ("Small" in st) or (pidx == 0):
                obsA.setdefault(fid, {})[cid] = uv
            elif ("Large" in st) or (pidx == 1):
                obsB.setdefault(fid, {})[cid] = uv

    frames = sorted(frames)
    if max_frames and max_frames > 0:
        frames = frames[:max_frames]

    return {"obsA": obsA, "obsB": obsB, "frames": frames}


# =========================
# Initial build
# =========================
cam_init, cam_to_window, window_planes_init, window_media = load_cam_set(init_cam_dir)
dataset = load_obs_dataset(obs_csv, MAX_FRAMES)

cam_ids = sorted(cam_init.keys())
win_ids = sorted(window_planes_init.keys())
fids = [fid for fid in dataset["frames"] if fid in dataset["obsA"] and fid in dataset["obsB"]]
fid_to_idx = {fid:i for i, fid in enumerate(fids)}
nF = len(fids)

print(f"cams={len(cam_ids)}, windows={len(win_ids)}, frames={nF}")

class _Base: pass
base = _Base()
base.image_size = IMG_SIZE
base.dist_coeff_num = 0

cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_init,
    window_media=window_media,
    cam_to_window=cam_to_window,
    window_planes=window_planes_init
)

# gauge fix: keep one reference cam fixed
ref_cam = cam_ids[0]
opt_cam_ids = [c for c in cam_ids if c != ref_cam]
print("reference cam fixed:", ref_cam)

# flatten observations
obs_rows = []
for fid in fids:
    for cid, uv in dataset["obsA"].get(fid, {}).items():
        if cid in cam_ids:
            obs_rows.append((fid, cid, 0, uv))
    for cid, uv in dataset["obsB"].get(fid, {}).items():
        if cid in cam_ids:
            obs_rows.append((fid, cid, 1, uv))
n_obs = len(obs_rows)
print("observations:", n_obs)

# =========================
# One-time triangulation init
# =========================
def triangulate_init(cams_cpp_local, cam_params_local, planes_local):
    XA = np.zeros((nF, 3), dtype=float)
    XB = np.zeros((nF, 3), dtype=float)

    def tri_endpoint(obs_key, endpoint_flag, Xout):
        per_cam = {}
        for i, fid in enumerate(fids):
            obs_f = dataset[obs_key].get(fid, {})
            for cid, uv in obs_f.items():
                if cid in cams_cpp_local:
                    per_cam.setdefault(cid, []).append((i, fid, uv))

        rays_by_i = {i: [] for i in range(nF)}
        for cid, items in per_cam.items():
            wid = cam_to_window[cid]
            uv_list = [uv for _, _, uv in items]
            meta = [{"cam_id":cid, "window_id":wid, "frame_id":fid, "endpoint":"A" if endpoint_flag==0 else "B"} for _, fid, _ in items]
            rays = build_pinplate_rays_cpp_batch(cams_cpp_local[cid], uv_list, meta_list=meta)
            for (i, _, _), r in zip(items, rays):
                if r.valid:
                    rays_by_i[i].append(r)

        for i in range(nF):
            rays = rays_by_i[i]
            if len(rays) < 2:
                continue
            A = np.zeros((3,3)); b = np.zeros(3)
            for r in rays:
                d = r.d / (np.linalg.norm(r.d) + 1e-12)
                o = r.o
                P = np.eye(3) - np.outer(d, d)
                A += P
                b += P @ o
            Xout[i] = np.linalg.solve(A + 1e-9*np.eye(3), b)

    tri_endpoint("obsA", 0, XA)
    tri_endpoint("obsB", 1, XB)

    for i in range(nF):
        if not np.all(np.isfinite(XA[i])):
            XA[i] = np.array([0,0,0], dtype=float)
        if not np.all(np.isfinite(XB[i])):
            XB[i] = XA[i] + np.array([0,0,WAND_LEN_MM], dtype=float)

    return XA, XB

XA0, XB0 = triangulate_init(cams_cpp, cam_init, window_planes_init)

# =========================
# B case: rotate-align BEFORE optimization
# =========================
pts_for_align = np.vstack([XA0, XB0])  # (2*nF, 3)
cam_aligned, planes_aligned, pts_aligned, R_align, t_shift = align_world_y_to_plane_intersection(
    window_planes_init,
    cam_init,
    points_3d=pts_for_align
)

# normalize plane array types
planes_aligned = {
    int(w): {
        **pl,
        "plane_pt": np.asarray(pl["plane_pt"], dtype=np.float64),
        "plane_n": np.asarray(pl["plane_n"], dtype=np.float64),
        "initialized": True
    }
    for w, pl in planes_aligned.items()
}
cam_aligned = {int(c): np.asarray(p, dtype=np.float64) for c, p in cam_aligned.items()}

# reuse aligned points as point init if available
if (pts_aligned is not None) and (len(pts_aligned) == 2*nF):
    pts_aligned = np.asarray(pts_aligned, dtype=float)
    XA0 = pts_aligned[:nF].copy()
    XB0 = pts_aligned[nF:].copy()

# rebuild C++ cams with aligned params
cams_cpp = CppCameraFactory.init_cams_cpp_in_memory(
    base=base,
    cam_params=cam_aligned,
    window_media=window_media,
    cam_to_window=cam_to_window,
    window_planes=planes_aligned
)

print("[B] alignment applied before SBA")
print("R_align=\n", R_align)
print("t_shift=", t_shift)

# =========================
# Parameter layout
# =========================
# optimize cam extrinsics + planes + points
# keep intrinsics fixed
cam_intr_fixed = {cid: cam_aligned[cid][6:11].copy() for cid in cam_ids}
cam_r0 = {cid: cam_aligned[cid][:3].copy() for cid in cam_ids}
cam_t0 = {cid: cam_aligned[cid][3:6].copy() for cid in cam_ids}
pl_n0  = {wid: planes_aligned[wid]["plane_n"].copy() for wid in win_ids}
pl_pt0 = {wid: planes_aligned[wid]["plane_pt"].copy() for wid in win_ids}

cam_dim = 6 * len(opt_cam_ids)
win_dim = 5 * len(win_ids)    # plane_pt(3) + a,b(2)
pts_dim = 6 * nF              # XA(3)+XB(3)

off_cam = 0
off_win = off_cam + cam_dim
off_pts = off_win + win_dim
n_params = off_pts + pts_dim

# residuals: obs + wand length (NO X prior)
off_obs = 0
off_len = off_obs + n_obs
m_res = off_len + nF

print("n_params:", n_params, "n_residuals:", m_res)

def pack_x0():
    x = np.zeros(n_params, dtype=float)
    k = off_cam
    for cid in opt_cam_ids:
        x[k:k+3] = cam_r0[cid]
        x[k+3:k+6] = cam_t0[cid]
        k += 6

    k = off_win
    for wid in win_ids:
        x[k:k+3] = pl_pt0[wid]
        x[k+3] = 0.0
        x[k+4] = 0.0
        k += 5

    k = off_pts
    for i in range(nF):
        x[k:k+3] = XA0[i]
        x[k+3:k+6] = XB0[i]
        k += 6

    return x

def unpack_x(x):
    # cameras
    cams = {}
    for cid in cam_ids:
        p = cam_aligned[cid].copy()
        if cid == ref_cam:
            p[:3] = cam_r0[cid]
            p[3:6] = cam_t0[cid]
        else:
            j = opt_cam_ids.index(cid)
            b = off_cam + 6*j
            p[:3] = x[b:b+3]
            p[3:6] = x[b+3:b+6]
        p[6:11] = cam_intr_fixed[cid]
        cams[cid] = p

    # planes
    planes = {}
    for j, wid in enumerate(win_ids):
        b = off_win + 5*j
        pt = x[b:b+3]
        a = x[b+3]
        c = x[b+4]
        n = update_normal_tangent(pl_n0[wid], a, c)
        planes[wid] = {
            "plane_pt": pt,
            "plane_n": n,
            "initialized": True
        }

    # points
    XA = np.zeros((nF,3), dtype=float)
    XB = np.zeros((nF,3), dtype=float)
    for i in range(nF):
        b = off_pts + 6*i
        XA[i] = x[b:b+3]
        XB[i] = x[b+3:b+6]

    return cams, planes, XA, XB

# =========================
# Jacobian sparsity
# =========================
Jsp = lil_matrix((m_res, n_params), dtype=np.int8)

# obs rows: depend on cam + plane + point
for ridx, (fid, cid, endpoint, uv) in enumerate(obs_rows):
    if cid != ref_cam:
        j = opt_cam_ids.index(cid)
        cb = off_cam + 6*j
        Jsp[ridx, cb:cb+6] = 1

    wid = cam_to_window[cid]
    jw = win_ids.index(wid)
    wb = off_win + 5*jw
    Jsp[ridx, wb:wb+5] = 1

    i = fid_to_idx[fid]
    pb = off_pts + 6*i
    if endpoint == 0:
        Jsp[ridx, pb:pb+3] = 1
    else:
        Jsp[ridx, pb+3:pb+6] = 1

# length rows: depend on XA/XB of frame
for i in range(nF):
    ridx = off_len + i
    pb = off_pts + 6*i
    Jsp[ridx, pb:pb+6] = 1

Jsp = Jsp.tocsr()
print("jac_sparsity:", Jsp.shape, "nnz:", Jsp.nnz)

# =========================
# Scales
# =========================
avg_f = np.mean([cam_aligned[c][6] for c in cam_ids])
z_guess = []
for cid in cam_ids:
    p = cam_aligned[cid]
    R, _ = cv2.Rodrigues(p[:3])
    C = camera_center(R, p[3:6])
    z_guess.append(abs(C[2]) + 1e-6)
avg_z = np.mean(z_guess)

sigma_ray = 1.0 * (avg_z / avg_f)     # 1 px -> mm
sigma_wand = 0.02 * WAND_LEN_MM
print(f"sigma_ray={sigma_ray:.6f} mm, sigma_wand={sigma_wand:.6f} mm")

# =========================
# Residual function + progress every 10 fve
# =========================
state = {"fve": 0, "t0": time.time()}

def residual_fun(x):
    cams, planes, XA, XB = unpack_x(x)

    # sync C++ cameras
    for cid in cam_ids:
        kw = CppSyncAdapter.build_update_kwargs(
            cam_params=cams,
            window_planes=planes,
            window_media=window_media,
            cam_to_window=cam_to_window,
            cam_id=cid
        )
        CppSyncAdapter.apply(cams_cpp, cid, kw)

    # batch rays
    perA, perB = {}, {}
    for fid, cid, endpoint, uv in obs_rows:
        if endpoint == 0:
            perA.setdefault(cid, []).append((fid, uv))
        else:
            perB.setdefault(cid, []).append((fid, uv))

    lookupA, lookupB = {}, {}
    for cid, items in perA.items():
        wid = cam_to_window[cid]
        uv_list = [uv for _, uv in items]
        meta = [{"cam_id":cid, "window_id":wid, "frame_id":fid, "endpoint":"A"} for fid, _ in items]
        rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta)
        for (fid, _), r in zip(items, rays):
            lookupA[(fid, cid)] = r

    for cid, items in perB.items():
        wid = cam_to_window[cid]
        uv_list = [uv for _, uv in items]
        meta = [{"cam_id":cid, "window_id":wid, "frame_id":fid, "endpoint":"B"} for fid, _ in items]
        rays = build_pinplate_rays_cpp_batch(cams_cpp[cid], uv_list, meta_list=meta)
        for (fid, _), r in zip(items, rays):
            lookupB[(fid, cid)] = r

    res = np.zeros(m_res, dtype=float)

    S_ray = 0.0
    N_ray = 0
    for ridx, (fid, cid, endpoint, uv) in enumerate(obs_rows):
        i = fid_to_idx[fid]
        X = XA[i] if endpoint == 0 else XB[i]
        r = lookupA.get((fid, cid)) if endpoint == 0 else lookupB.get((fid, cid))
        if (r is not None) and r.valid:
            d = point_to_ray_dist(X, r.o, r.d)
            res[off_obs + ridx] = d / sigma_ray
            S_ray += d*d
            N_ray += 1
        else:
            res[off_obs + ridx] = 100.0 / sigma_ray

    S_len = 0.0
    for i in range(nF):
        dl = np.linalg.norm(XB[i] - XA[i]) - WAND_LEN_MM
        res[off_len + i] = dl / sigma_wand
        S_len += dl*dl

    state["fve"] += 1
    if state["fve"] % 10 == 0:
        ray_rmse = np.sqrt(S_ray / max(1, N_ray))
        len_rmse = np.sqrt(S_len / max(1, nF))
        dt = time.time() - state["t0"]
        print(f"[fve~{state['fve']:4d}] ray={ray_rmse:.6f} mm | len={len_rmse:.6f} mm | elapsed={dt:.1f}s")

    return res


# =========================
# Run SBA (B only)
# =========================
x0 = pack_x0()
print("\nStart SBA (B: pre-rot-align + no X regularization)...")

t0 = time.time()
res = least_squares(
    residual_fun,
    x0,
    method="trf",
    jac_sparsity=Jsp,
    loss=LOSS,
    ftol=FTOL,
    xtol=XTOL,
    gtol=GTOL,
    max_nfev=MAX_NFEV,
    verbose=0
)
dt = time.time() - t0

print("\n=== SBA B Done ===")
print("status :", res.status)
print("message:", res.message)
print("nfev   :", res.nfev)
print("time(s):", round(dt, 2))
print("fve calls:", state["fve"])


cams=4, windows=2, frames=2500

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
reference cam fixed: 0
observations: 20000
[Coordinate Alignment] Intersection line direction: [-0.2016, 0.9761, -0.0815]
[Coordinate Alignment] Intersection line direction: [-0.2016, 0.9761, -0.0815]
[ALIGN CHECK] Y-Axis (Intersection): [-4.27069348e-18  1.00000000e+00  1.89668606e-17]
[ALIGN CHECK] Z-Axis (Plane 0 Normal): [-2.01129160e-17  1.89668606e-17  1.00000000e+00]
[Coordinate Alignment] Centering World at Cloud Centroid: [  4.18181678  -0.68259398 184.97554366]
[Coordinate Alignment] Applied rotation and translation to align coordinates.

Initializing C++ Camera objects in-memory...
  Initialized 4 C++ Camera objects.
[B] alignment applied before SBA
R_align=
 [[ 0.97721405  0.19478644 -0.08432645]
 [-0.20159031  0.97607522 -0.08147701]
 [ 0.06643834  0.09661988  0.99310148]]
t_shift= [  -4.18181678    0.68259398 -184.97554366]
n_params: 15028 n_residuals: 22500
ja

Synthetic test

In [2]:
import os, csv, json, math
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

import pyopenlpt as lpt

# =========================================================
# Global settings
# =========================================================
OUT_ROOT = Path(r"J:\Refraction_test")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

IMG_W, IMG_H = 1280, 800
FOCAL = 9000.0
CX, CY = IMG_W / 2.0, IMG_H / 2.0

WAND_LEN_MM = 10.0
R_SMALL_MM = 1.5
R_LARGE_MM = 2.0
METRIC_CONST = 1.0

N_OBJ, N_WIN, N_AIR = 1.333, 1.49, 1.0
THICK_MM = 10.0
REFRACT_ARRAY = [N_OBJ, N_WIN, N_AIR]   # farthest->nearest
W_ARRAY = [THICK_MM]

SEED_BASE = 20260223
N_FRAMES_TARGET = 2000
MAX_FRAME_TRIES = 1_200_000
HALFSPACE_MARGIN_MM = 2.0
D_PLANE_MM = 120.0

DIST_MAP = {
    "Near": 550.0,
    "Mid": 800.0,
    "Far": 1100.0,
}

# ---------------------------------------------------------
# 30-case matrix
# G is adjacent face inner angle
# A uses 15/30/45 now
# ---------------------------------------------------------
CASES = [
    {"id": 1,  "P": 1, "C": [2],       "G": None, "A": "U15", "D": "Near"},
    {"id": 2,  "P": 1, "C": [2],       "G": None, "A": "U45", "D": "Far"},
    {"id": 3,  "P": 1, "C": [3],       "G": None, "A": "U30", "D": "Mid"},
    {"id": 4,  "P": 1, "C": [3],       "G": None, "A": "M3",  "D": "Near"},
    {"id": 5,  "P": 1, "C": [4],       "G": None, "A": "U15", "D": "Far"},
    {"id": 6,  "P": 1, "C": [4],       "G": None, "A": "M2",  "D": "Mid"},

    {"id": 7,  "P": 2, "C": [2,2],     "G": 60,   "A": "U30", "D": "Near"},
    {"id": 8,  "P": 2, "C": [2,2],     "G": 120,  "A": "M2",  "D": "Far"},
    {"id": 9,  "P": 2, "C": [3,3],     "G": 90,   "A": "U30", "D": "Mid"},
    {"id": 10, "P": 2, "C": [3,3],     "G": 60,   "A": "M3",  "D": "Far"},
    {"id": 11, "P": 2, "C": [2,3],     "G": 90,   "A": "U15", "D": "Near"},
    {"id": 12, "P": 2, "C": [2,3],     "G": 120,  "A": "M2",  "D": "Mid"},
    {"id": 13, "P": 2, "C": [2,4],     "G": 60,   "A": "U45", "D": "Far"},
    {"id": 14, "P": 2, "C": [2,4],     "G": 90,   "A": "M3",  "D": "Near"},
    {"id": 15, "P": 2, "C": [3,4],     "G": 120,  "A": "U30", "D": "Mid"},
    {"id": 16, "P": 2, "C": [3,4],     "G": 60,   "A": "M2",  "D": "Far"},
    {"id": 17, "P": 2, "C": [4,4],     "G": 90,   "A": "U15", "D": "Mid"},
    {"id": 18, "P": 2, "C": [4,4],     "G": 120,  "A": "M3",  "D": "Near"},

    {"id": 19, "P": 3, "C": [2,2,2],   "G": 60,   "A": "U30", "D": "Near"},
    {"id": 20, "P": 3, "C": [2,2,2],   "G": 120,  "A": "M3",  "D": "Far"},
    {"id": 21, "P": 3, "C": [3,3,3],   "G": 90,   "A": "U30", "D": "Mid"},
    {"id": 22, "P": 3, "C": [3,3,3],   "G": 60,   "A": "M2",  "D": "Far"},
    {"id": 23, "P": 3, "C": [2,3,4],   "G": 90,   "A": "M3",  "D": "Near"},
    {"id": 24, "P": 3, "C": [2,3,4],   "G": 120,  "A": "U15", "D": "Mid"},
    {"id": 25, "P": 3, "C": [2,4,4],   "G": 60,   "A": "U45", "D": "Far"},
    {"id": 26, "P": 3, "C": [2,4,4],   "G": 90,   "A": "M2",  "D": "Near"},
    {"id": 27, "P": 3, "C": [3,2,4],   "G": 120,  "A": "M3",  "D": "Mid"},
    {"id": 28, "P": 3, "C": [3,2,4],   "G": 60,   "A": "U15", "D": "Far"},
    {"id": 29, "P": 3, "C": [4,3,2],   "G": 90,   "A": "M2",  "D": "Mid"},
    {"id": 30, "P": 3, "C": [4,3,2],   "G": 120,  "A": "U45", "D": "Near"},
]

# =========================================================
# Math helpers
# =========================================================
def nrm(v):
    n = np.linalg.norm(v)
    return v if n < 1e-12 else v / n

def basis_from_n(n):
    n = nrm(n)
    ref = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(n, ref)) > 0.95:
        ref = np.array([0.0, 1.0, 0.0])
    u = nrm(np.cross(n, ref))
    v = nrm(np.cross(n, u))
    return u, v

def look_at_origin(C):
    z = nrm(-C)  # optical axis camera->origin
    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(z, up)) > 0.98:
        up = np.array([0.0, 1.0, 0.0])
    x = nrm(np.cross(up, z))
    y = nrm(np.cross(z, x))
    R = np.vstack([x, y, z])  # world->cam
    t = -R @ C
    return R, t

def rodrigues_rvec(R):
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()

def angle_profile(name, n):
    base = {
        "U15": [15.0],
        "U30": [30.0],
        "U45": [45.0],
        "M2":  [15.0, 45.0],
        "M3":  [15.0, 30.0, 45.0],
    }[name]
    return [base[i % len(base)] for i in range(n)]

# =========================================================
# Geometry generation
# =========================================================
def build_planes(P, G_inner_deg, d_plane=D_PLANE_MM):
    """
    G_inner_deg = adjacent face inner angle.
    Adjacent normal angle = 180 - G_inner_deg.
    """
    if P == 1:
        phis = [0.0]
    else:
        dphi = np.deg2rad(180.0 - G_inner_deg)
        phis = [i * dphi for i in range(P)]

    planes = []
    for i, ph in enumerate(phis):
        n = nrm(np.array([np.cos(ph), np.sin(ph), 0.0], dtype=float))
        p = -d_plane * n
        planes.append({"id": i, "n": n, "pt_far": p, "d_plane": d_plane})
    return planes

def enforce_plane_normal_direction(planes):
    # origin should be object-side: n·(O-p)>0
    O = np.zeros(3)
    for pl in planes:
        if np.dot(pl["n"], O - pl["pt_far"]) < 0:
            pl["n"] = -pl["n"]

def build_cameras(case, planes, phi_plane):
    """
    phi_plane: per-plane azimuth phase offsets (len=P)
    """
    cams = []
    cam_id = 0
    Rcam = DIST_MAP[case["D"]]

    for pid, cnum in enumerate(case["C"]):
        n = planes[pid]["n"]
        u, v = basis_from_n(n)
        betas = angle_profile(case["A"], cnum)  # angle(optical_axis, normal)
        ph0 = phi_plane[pid]

        for j in range(cnum):
            phi = ph0 + 2*np.pi*j/max(cnum, 1)
            t_hat = nrm(np.cos(phi)*u + np.sin(phi)*v)

            beta = np.deg2rad(betas[j])
            d = nrm(np.cos(beta)*n + np.sin(beta)*t_hat)  # camera->origin axis

            C = -Rcam * d
            R, t = look_at_origin(C)
            beta_chk = np.degrees(np.arccos(np.clip(np.dot(d, n), -1.0, 1.0)))

            cams.append({
                "cam_id": cam_id,
                "plane_id": pid,
                "C": C,
                "R": R,
                "t": t,
                "beta_deg": float(beta_chk),
            })
            cam_id += 1

    return cams

# =========================================================
# Cam file writer
# =========================================================
def write_camfile(path, cam, plane):
    R, t, C = cam["R"], cam["t"], cam["C"]
    n, p = plane["n"], plane["pt_far"]
    rv = rodrigues_rvec(R)

    with open(path, "w", encoding="utf-8") as f:
        f.write("# Camera Model: (PINHOLE/POLYNOMIAL/PINPLATE)\nPINPLATE\n")
        f.write("# Camera Calibration Error:\n0,0\n")
        f.write("# Pose Calibration Error:\n0,0\n")
        f.write(f"# Image Size: (n_row,n_col)\n{IMG_H},{IMG_W}\n")
        f.write("# Camera Matrix:\n")
        f.write(f"{FOCAL:.8g} 0 {CX:.8g}\n")
        f.write(f"0 {FOCAL:.8g} {CY:.8g}\n")
        f.write("0 0 1\n")
        f.write("# Distortion Coefficients:\n0,0,0,0,0\n")
        f.write(f"# Rotation Vector:\n{rv[0]:.8g},{rv[1]:.8g},{rv[2]:.8g}\n")
        f.write("# Rotation Matrix:\n")
        for row in R:
            f.write(f"{row[0]:.8g} {row[1]:.8g} {row[2]:.8g}\n")
        f.write("# Inverse of Rotation Matrix:\n")
        for row in R.T:
            f.write(f"{row[0]:.8g} {row[1]:.8g} {row[2]:.8g}\n")
        f.write(f"# Translation Vector:\n{t[0]:.8g} {t[1]:.8g} {t[2]:.8g}\n")
        f.write(f"# Inverse of Translation Vector:\n{C[0]:.8g} {C[1]:.8g} {C[2]:.8g}\n")
        f.write("# Refractive plane reference point plane.pt (Farthest Interface)\n")
        f.write(f"{p[0]:.8g} {p[1]:.8g} {p[2]:.8g}\n")
        f.write("# Refractive plane normal plane.norm_vector (camera->object direction)\n")
        f.write(f"{n[0]:.8g} {n[1]:.8g} {n[2]:.8g}\n")
        f.write("# refract_array (ONE token, comma-separated, farthest->nearest: obj->win->air)\n")
        f.write(",".join(f"{x:.4f}" for x in REFRACT_ARRAY) + "\n")
        f.write("# w_array (ONE token, comma-separated, plate thicknesses in mm)\n")
        f.write(",".join(f"{x:.4f}" for x in W_ARRAY) + "\n")
        f.write("# proj_tol\n1e-6\n# proj_nmax\n50\n# lr (learning rate)\n0.1\n")
        f.write("# --- BEGIN_REFRACTION_META ---\n")
        f.write("# VERSION=2\n")
        f.write(f"# CAM_ID={cam['cam_id']}\n")
        f.write(f"# WINDOW_ID={cam['plane_id']}\n")
        f.write("# --- END_REFRACTION_META ---\n")

# =========================================================
# Visibility / FOV
# =========================================================
def project_batch(cam_obj, pts):
    pts_lpt = [lpt.Pt3D(float(p[0]), float(p[1]), float(p[2])) for p in pts]
    try:
        st = cam_obj.projectBatchStatus(pts_lpt, False)
    except TypeError:
        st = cam_obj.projectBatchStatus(pts_lpt)

    uv = np.full((len(pts), 2), np.nan, dtype=float)
    ok = np.zeros(len(pts), dtype=bool)
    for i, s in enumerate(st):
        if s and s[0]:
            u, v = float(s[1][0]), float(s[1][1])
            uv[i] = [u, v]
            ok[i] = (0 <= u < IMG_W) and (0 <= v < IMG_H)
    return uv, ok

def project_one(cam_obj, X):
    p = lpt.Pt3D(float(X[0]), float(X[1]), float(X[2]))
    try:
        st = cam_obj.projectStatus(p, False)
    except TypeError:
        st = cam_obj.projectStatus(p)
    if not st or not st[0]:
        return None, False
    u, v = float(st[1][0]), float(st[1][1])
    ok = (0 <= u < IMG_W) and (0 <= v < IMG_H)
    return np.array([u, v]), ok

def in_object_side_all_planes(X, planes, margin_mm):
    for pl in planes:
        if np.dot(pl["n"], X - pl["pt_far"]) <= margin_mm:
            return False
    return True

def collect_valid_voxels(cams_obj, planes, bmin, bmax, step, margin_mm):
    xs = np.arange(bmin[0], bmax[0] + 1e-9, step)
    ys = np.arange(bmin[1], bmax[1] + 1e-9, step)
    zs = np.arange(bmin[2], bmax[2] + 1e-9, step)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="xy")
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

    hs_ok = np.array([in_object_side_all_planes(p, planes, margin_mm) for p in pts], dtype=bool)
    if not np.any(hs_ok):
        return np.empty((0, 3)), {"n_vox": int(len(pts)), "n_halfspace": 0, "n_valid": 0, "step": float(step)}

    vis = hs_ok.copy()
    for cam in cams_obj:
        _, ok = project_batch(cam, pts)
        vis &= ok
        if not np.any(vis):
            break

    valid = pts[vis]
    stat = {
        "n_vox": int(len(pts)),
        "n_halfspace": int(np.sum(hs_ok)),
        "n_valid": int(len(valid)),
        "step": float(step),
    }
    return valid, stat

# =========================================================
# Point generation
# =========================================================
def radius_px(R, t, Xw, r_mm):
    Xc = R @ Xw + t
    z = float(Xc[2])
    if z <= 1e-9:
        return np.nan
    return FOCAL * r_mm / z

def sample_wand_points(cams, cams_obj, planes, valid_pts, rng, n_target, margin_mm, voxel_step):
    if len(valid_pts) == 0:
        return None, None, None, {"tries": 0, "accepted": 0, "reject_side": 0, "reject_vis": 0}

    A_list, B_list, rows = [], [], []
    tries, reject_side, reject_vis = 0, 0, 0

    while len(A_list) < n_target and tries < MAX_FRAME_TRIES:
        tries += 1
        c = valid_pts[rng.integers(0, len(valid_pts))] + rng.uniform(-0.35*voxel_step, 0.35*voxel_step, size=3)
        u = nrm(rng.normal(size=3))
        A = c - 0.5 * WAND_LEN_MM * u
        B = c + 0.5 * WAND_LEN_MM * u

        if (not in_object_side_all_planes(A, planes, margin_mm)) or (not in_object_side_all_planes(B, planes, margin_mm)):
            reject_side += 1
            continue

        ok_all = True
        uvA_by, uvB_by = {}, {}
        for cm, co in zip(cams, cams_obj):
            uvA, okA = project_one(co, A)
            uvB, okB = project_one(co, B)
            if not (okA and okB):
                ok_all = False
                break
            uvA_by[cm["cam_id"]] = uvA
            uvB_by[cm["cam_id"]] = uvB

        if not ok_all:
            reject_vis += 1
            continue

        fid = len(A_list)
        A_list.append(A); B_list.append(B)

        for cm in cams:
            cid = cm["cam_id"]
            uA, vA = uvA_by[cid]
            uB, vB = uvB_by[cid]
            rA = radius_px(cm["R"], cm["t"], A, R_SMALL_MM)
            rB = radius_px(cm["R"], cm["t"], B, R_LARGE_MM)
            rows.append([fid, cid, "Filtered_Small", 0, f"{uA:.6f}", f"{vA:.6f}", f"{rA:.6f}", f"{METRIC_CONST:.6f}"])
            rows.append([fid, cid, "Filtered_Large", 1, f"{uB:.6f}", f"{vB:.6f}", f"{rB:.6f}", f"{METRIC_CONST:.6f}"])

    if len(A_list) < n_target:
        return None, None, None, {
            "tries": int(tries),
            "accepted": int(len(A_list)),
            "reject_side": int(reject_side),
            "reject_vis": int(reject_vis),
        }

    return np.array(A_list), np.array(B_list), rows, {
        "tries": int(tries),
        "accepted": int(len(A_list)),
        "reject_side": int(reject_side),
        "reject_vis": int(reject_vis),
    }

# =========================================================
# Plot
# =========================================================
def save_scene_plot(path_png, planes, cams, A_pts, B_pts):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    s = 120.0
    for pl in planes:
        n, p0 = pl["n"], pl["pt_far"]
        u, v = basis_from_n(n)
        q = np.array([p0+s*u+s*v, p0+s*u-s*v, p0-s*u-s*v, p0-s*u+s*v])
        ax.plot(np.r_[q[:,0], q[0,0]], np.r_[q[:,1], q[0,1]], np.r_[q[:,2], q[0,2]], "-", lw=1.2)

    C = np.array([c["C"] for c in cams])
    D = np.array([nrm(-c["C"]) for c in cams])
    ax.scatter(C[:,0], C[:,1], C[:,2], c="r", s=24, label="Cameras")
    ax.quiver(C[:,0], C[:,1], C[:,2], D[:,0], D[:,1], D[:,2], length=80, normalize=True, color="r")

    ax.scatter(A_pts[:,0], A_pts[:,1], A_pts[:,2], c="b", s=5, label="A")
    ax.scatter(B_pts[:,0], B_pts[:,1], B_pts[:,2], c="g", s=5, label="B")
    ax.scatter([0], [0], [0], c="k", marker="x", s=40, label="Origin")

    ax.set_xlabel("X(mm)"); ax.set_ylabel("Y(mm)"); ax.set_zlabel("Z(mm)")
    ax.set_title("Synthetic Scene")
    ax.legend(loc="upper right")
    ax.set_box_aspect([1,1,1])
    plt.tight_layout()
    plt.savefig(path_png, dpi=170)
    plt.close(fig)

# =========================================================
# Layout search
# =========================================================
def search_best_layout(case, retries, rng_base):
    """
    Search phi_plane per plane to maximize valid voxel count.
    """
    P = case["P"]
    best = None

    # coarse region for quick score
    coarse_bmin = np.array([-280.0, -280.0, -280.0])
    coarse_bmax = np.array([ 280.0,  280.0,  280.0])

    for k in range(retries):
        rng = np.random.default_rng(rng_base + k)

        planes = build_planes(case["P"], case["G"] if case["G"] is not None else 90, d_plane=D_PLANE_MM)
        enforce_plane_normal_direction(planes)

        # per-plane phase offsets
        phi_plane = rng.uniform(0, 2*np.pi, size=P)

        cams = build_cameras(case, planes, phi_plane)

        # temporary write/load cameras in memory from formula is enough for score:
        # use pinhole projection score first (fast proxy)
        # Here still using pyopenlpt scoring for fidelity:
        tmp_dir = OUT_ROOT / "_tmp_layout_eval"
        tmp_dir.mkdir(parents=True, exist_ok=True)
        for c in cams:
            write_camfile(tmp_dir / f"cam{c['cam_id']}.txt", c, planes[c["plane_id"]])
        cams_obj = [lpt.Camera(str(tmp_dir / f"cam{c['cam_id']}.txt")) for c in cams]

        valid, stat = collect_valid_voxels(
            cams_obj, planes,
            bmin=coarse_bmin, bmax=coarse_bmax,
            step=16.0, margin_mm=HALFSPACE_MARGIN_MM
        )
        score = stat["n_valid"]

        if (best is None) or (score > best["score"]):
            best = {
                "score": score,
                "planes": planes,
                "cams": cams,
                "phi_plane": phi_plane.tolist(),
                "coarse_stat": stat
            }

    return best

# =========================================================
# Main run
# =========================================================
summary_rows = []
failed_rows = []

for case in CASES[25:]:
    case_id = case["id"]
    case_dir = OUT_ROOT / f"case_{case_id:03d}"
    cam_dir = case_dir / "Synthetic_camFile"
    case_dir.mkdir(parents=True, exist_ok=True)
    cam_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Case {case_id:03d} ===")

    best_layout = search_best_layout(case, retries=240, rng_base=SEED_BASE + 10000*case_id)
    if best_layout is None:
        reason = "layout search failed"
        failed_rows.append([case_id, reason])
        with open(case_dir / "case_meta.json", "w", encoding="utf-8") as f:
            json.dump({"case": case, "status": "failed", "reason": reason}, f, indent=2)
        print(f"[FAIL] {reason}")
        continue

    planes = best_layout["planes"]
    cams = best_layout["cams"]

    # write final cam files
    for c in cams:
        write_camfile(cam_dir / f"cam{c['cam_id']}.txt", c, planes[c["plane_id"]])

    cams_obj = [lpt.Camera(str(cam_dir / f"cam{c['cam_id']}.txt")) for c in cams]

    # refined valid voxels
    refined_bmin = np.array([-320.0, -320.0, -320.0])
    refined_bmax = np.array([ 320.0,  320.0,  320.0])
    valid_pts, fov_stat = collect_valid_voxels(
        cams_obj, planes,
        bmin=refined_bmin, bmax=refined_bmax,
        step=8.0, margin_mm=HALFSPACE_MARGIN_MM
    )

    if len(valid_pts) == 0:
        reason = f"empty common FOV (best coarse score={best_layout['score']})"
        failed_rows.append([case_id, reason])
        with open(case_dir / "case_meta.json", "w", encoding="utf-8") as f:
            json.dump({
                "case": case,
                "status": "failed",
                "reason": reason,
                "layout_search": {
                    "best_phi_plane": best_layout["phi_plane"],
                    "best_coarse_stat": best_layout["coarse_stat"]
                }
            }, f, indent=2)
        print(f"[FAIL] {reason}")
        continue

    rng = np.random.default_rng(SEED_BASE + 20000*case_id)
    A_pts, B_pts, rows, gen_stat = sample_wand_points(
        cams, cams_obj, planes, valid_pts, rng,
        n_target=N_FRAMES_TARGET, margin_mm=HALFSPACE_MARGIN_MM, voxel_step=fov_stat["step"]
    )

    if A_pts is None:
        reason = f"insufficient frames: {gen_stat}"
        failed_rows.append([case_id, reason])
        with open(case_dir / "case_meta.json", "w", encoding="utf-8") as f:
            json.dump({
                "case": case,
                "status": "failed",
                "reason": reason,
                "layout_search": {
                    "best_phi_plane": best_layout["phi_plane"],
                    "best_coarse_stat": best_layout["coarse_stat"]
                },
                "fov": {
                    **fov_stat,
                    "bbox_min": valid_pts.min(axis=0).tolist(),
                    "bbox_max": valid_pts.max(axis=0).tolist()
                }
            }, f, indent=2)
        print(f"[FAIL] {reason}")
        continue

    # save csv
    csv_path = case_dir / "synthetic_wand_points.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Frame", "Camera", "Status", "PointIdx", "X", "Y", "Radius", "Metric"])
        w.writerows(rows)

    # save plot
    plot_path = case_dir / "scene_overview.png"
    save_scene_plot(plot_path, planes, cams, A_pts, B_pts)

    # save meta
    meta = {
        "case": case,
        "status": "ok",
        "focal_px": FOCAL,
        "image_size": [IMG_H, IMG_W],
        "wand": {
            "length_mm": WAND_LEN_MM,
            "radius_small_mm": R_SMALL_MM,
            "radius_large_mm": R_LARGE_MM,
            "metric": METRIC_CONST,
            "n_frames": int(len(A_pts)),
        },
        "constraints": {
            "all_camera_visible": True,
            "halfspace_margin_mm": HALFSPACE_MARGIN_MM,
            "angle_definition": "camera optical axis vs plane normal",
            "adjacent_plane_angle_definition": "inner angle G"
        },
        "layout_search": {
            "best_phi_plane": best_layout["phi_plane"],
            "best_coarse_stat": best_layout["coarse_stat"],
            "best_coarse_score_n_valid": best_layout["score"]
        },
        "fov": {
            **fov_stat,
            "bbox_min": valid_pts.min(axis=0).tolist(),
            "bbox_max": valid_pts.max(axis=0).tolist()
        },
        "generation": gen_stat,
        "planes": [
            {"id": p["id"], "n": p["n"].tolist(), "pt_far": p["pt_far"].tolist(), "d_from_origin_mm": p["d_plane"]}
            for p in planes
        ],
        "cameras": [
            {"cam_id": c["cam_id"], "plane_id": c["plane_id"], "C_world": c["C"].tolist(), "beta_deg": c["beta_deg"]}
            for c in cams
        ],
        "outputs": {
            "cam_dir": str(cam_dir),
            "csv": str(csv_path),
            "plot": str(plot_path),
        }
    }
    with open(case_dir / "case_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    summary_rows.append([case_id, len(cams), len(A_pts), str(csv_path)])
    print(f"[OK] case_{case_id:03d} cams={len(cams)} frames={len(A_pts)} valid_vox={fov_stat['n_valid']}")

# summary files
manifest_ok = OUT_ROOT / "cases_manifest.csv"
with open(manifest_ok, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["case_id", "n_cams", "n_frames", "csv"])
    w.writerows(summary_rows)

manifest_fail = OUT_ROOT / "cases_failed.csv"
with open(manifest_fail, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["case_id", "reason"])
    w.writerows(failed_rows)

print("\nDone.")
print("  OK manifest   :", manifest_ok)
print("  Failed manifest:", manifest_fail)



=== Case 026 ===
[OK] case_026 cams=10 frames=2000 valid_vox=2

=== Case 027 ===
[OK] case_027 cams=9 frames=2000 valid_vox=408

=== Case 028 ===
[OK] case_028 cams=9 frames=2000 valid_vox=2476

=== Case 029 ===
[OK] case_029 cams=9 frames=2000 valid_vox=275

=== Case 030 ===
[FAIL] empty common FOV (best coarse score=0)

Done.
  OK manifest   : J:\Refraction_test\cases_manifest.csv
  Failed manifest: J:\Refraction_test\cases_failed.csv


Verify the synthetic data

In [1]:
"""
Synthetic数据几何残差验证（无GT版本）
- 直接遍历 case_* 目录
- 直接用 lpt.Camera(file_path) 读取 cam*.txt
- 用 lineOfSightBatchStatus + 多光线最小二乘重构3D
- 几何残差定义：重构点到各相机光线的距离（mm），取该点的 max residual
- 额外检查：每帧 wand length 误差 | ||XB-XA|| - WAND_LEN_MM |
- 判定：
  1) 每个点 max residual < 0.001 mm
  2) 每帧 wand length 绝对误差 < 0.001 mm
  3) case 内所有点和所有帧均通过才算 case 通过
"""

from __future__ import annotations

import csv
import math
import re
from pathlib import Path
from collections import defaultdict
from typing import Dict, Tuple, List, Any

import numpy as np
import pyopenlpt as lpt


ROOT = Path(r"J:\Refraction_test")
THRESH_MM = 1e-3          # point-to-ray max residual threshold (mm)
WAND_LEN_MM = 10.0        # target wand length (mm)
WAND_THRESH_MM = 1e-3     # wand length abs error threshold (mm)


def make_pt2d(uv: Tuple[float, float]):
    return lpt.Pt2D(float(uv[0]), float(uv[1]))


def status_valid(v: str) -> bool:
    s = str(v).strip().lower()
    if s in {"1", "true", "ok", "valid", "yes"}:
        return True
    if s in {"0", "false", "invalid", "no", ""}:
        return False
    return True


def parse_cam_id(path: Path) -> int:
    m = re.match(r"cam(\d+)\.txt$", path.name, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f"Invalid camera filename: {path.name}")
    return int(m.group(1))


def load_cameras(cam_dir: Path) -> Dict[int, Any]:
    cams: Dict[int, Any] = {}
    cam_files = sorted(cam_dir.glob("cam*.txt"), key=lambda p: parse_cam_id(p))
    for f in cam_files:
        cid = parse_cam_id(f)
        cams[cid] = lpt.Camera(str(f))  # 直接读取cam*.txt
    return cams


def load_obs(csv_path: Path) -> Dict[Tuple[int, int], Dict[int, Tuple[float, float]]]:
    # key = (frame, point_idx), value = {cam_id: (u,v)}
    obs = defaultdict(dict)
    with csv_path.open("r", newline="", encoding="utf-8") as f:
        rd = csv.DictReader(f)
        required = {"Frame", "Camera", "Status", "PointIdx", "X", "Y"}
        if not required.issubset(set(rd.fieldnames or [])):
            raise RuntimeError(f"{csv_path} missing required columns: {required}")
        for row in rd:
            if not status_valid(row["Status"]):
                continue
            frame = int(row["Frame"])
            cam = int(row["Camera"])
            pidx = int(row["PointIdx"])
            u = float(row["X"])
            v = float(row["Y"])
            obs[(frame, pidx)][cam] = (u, v)
    return obs


def triangulate_from_rays(rays: List[Tuple[np.ndarray, np.ndarray]]):
    # rays: [(o,d), ...], d assumed normalized
    if len(rays) < 2:
        return np.zeros(3), np.inf, False, "insufficient_valid_rays"

    A = np.zeros((3, 3), dtype=np.float64)
    b = np.zeros(3, dtype=np.float64)
    I = np.eye(3, dtype=np.float64)

    for o, d in rays:
        d = d.reshape(3, 1)
        P = I - d @ d.T
        A += P
        b += P @ o

    try:
        cond = np.linalg.cond(A)
        if cond < 1e12:
            X = np.linalg.solve(A, b)
            return X, cond, True, ""
        X, _, rank, _ = np.linalg.lstsq(A, b, rcond=None)
        return X, cond, False, f"ill_conditioned(rank={rank}, cond={cond:.2e})"
    except Exception as e:
        return np.zeros(3), np.inf, False, f"linalg_error: {e}"


def point_to_ray_dist_halfline(X: np.ndarray, o: np.ndarray, d: np.ndarray) -> float:
    d = d / (np.linalg.norm(d) + 1e-12)
    v = X - o
    t = float(np.dot(v, d))
    if t >= 0.0:
        return float(np.linalg.norm(v - t * d))
    return float(np.linalg.norm(v))


def verify_case(case_dir: Path, thresh_mm: float):
    cam_dir = case_dir / "Synthetic_camFile"
    csv_file = case_dir / "synthetic_wand_points.csv"

    if not cam_dir.exists() or not csv_file.exists():
        return [], [], {
            "case": case_dir.name,
            "status": "SKIP",
            "reason": "missing Synthetic_camFile or synthetic_wand_points.csv",
            "total_points": 0,
            "failed_points": 0,
            "max_err_mm": "",
            "wand_failed_frames": 0,
            "wand_max_abs_err_mm": "",
            "pass": False,
        }

    cams = load_cameras(cam_dir)
    if len(cams) < 2:
        return [], [], {
            "case": case_dir.name,
            "status": "SKIP",
            "reason": f"not_enough_cameras({len(cams)})",
            "total_points": 0,
            "failed_points": 0,
            "max_err_mm": "",
            "wand_failed_frames": 0,
            "wand_max_abs_err_mm": "",
            "pass": False,
        }

    obs = load_obs(csv_file)

    point_rows = []
    failed = 0
    max_err = -np.inf

    # frame -> endpoint -> reconstructed point
    recon_by_frame = defaultdict(dict)

    for (frame, pidx), cam_uv in sorted(obs.items()):
        rays = []
        invalid_los = 0

        for cid, uv in cam_uv.items():
            cam = cams.get(cid, None)
            if cam is None:
                continue

            try:
                st = cam.lineOfSightBatchStatus([make_pt2d(uv)])
            except Exception:
                invalid_los += 1
                continue

            if not st:
                invalid_los += 1
                continue

            ok, line, _ = st[0]
            if not ok:
                invalid_los += 1
                continue

            o = np.array([line.pt[0], line.pt[1], line.pt[2]], dtype=np.float64)
            d = np.array([line.unit_vector[0], line.unit_vector[1], line.unit_vector[2]], dtype=np.float64)
            if not np.all(np.isfinite(o)) or not np.all(np.isfinite(d)):
                invalid_los += 1
                continue

            dn = np.linalg.norm(d)
            if dn < 1e-12:
                invalid_los += 1
                continue

            d = d / dn
            rays.append((o, d))

        if len(rays) < 2:
            err = math.inf
            ok = False
            reason = "insufficient_valid_rays"
        else:
            X, cond, tri_ok, tri_reason = triangulate_from_rays(rays)
            if not np.all(np.isfinite(X)):
                err = math.inf
                ok = False
                reason = "triangulate_nonfinite"
            else:
                dists = [point_to_ray_dist_halfline(X, o, d) for (o, d) in rays]
                err = float(np.max(dists)) if dists else math.inf
                ok = bool(tri_ok) and (err < thresh_mm)
                reason = "" if ok else (tri_reason if tri_reason else "residual_over_threshold")
                # save reconstructed endpoint for wand check
                recon_by_frame[frame][pidx] = X.copy()

        if np.isfinite(err):
            max_err = max(max_err, err)

        if not ok:
            failed += 1

        point_rows.append({
            "case": case_dir.name,
            "frame": frame,
            "point_idx": pidx,
            "num_obs": len(cam_uv),
            "num_valid_rays": len(rays),
            "invalid_los": invalid_los,
            "max_residual_mm": err,
            "pass": ok,
            "reason": reason,
        })

    # --- Wand length verification per frame ---
    wand_rows = []
    wand_failed = 0
    wand_max_err = -np.inf

    for frame in sorted(recon_by_frame.keys()):
        pts = recon_by_frame[frame]
        if 0 not in pts or 1 not in pts:
            wand_err = math.inf
            wand_ok = False
            wand_reason = "missing_endpoint"
        else:
            L = float(np.linalg.norm(pts[1] - pts[0]))
            wand_err = abs(L - WAND_LEN_MM)
            wand_ok = (wand_err < WAND_THRESH_MM)
            wand_reason = "" if wand_ok else "wand_length_over_threshold"

        if np.isfinite(wand_err):
            wand_max_err = max(wand_max_err, wand_err)
        if not wand_ok:
            wand_failed += 1

        wand_rows.append({
            "case": case_dir.name,
            "frame": frame,
            "wand_abs_err_mm": wand_err,
            "pass": wand_ok,
            "reason": wand_reason,
        })

    total = len(point_rows)
    case_pass = (total > 0 and failed == 0 and wand_failed == 0)

    case_row = {
        "case": case_dir.name,
        "status": "OK",
        "reason": "",
        "total_points": total,
        "failed_points": failed,
        "max_err_mm": (max_err if np.isfinite(max_err) else ""),
        "wand_failed_frames": wand_failed,
        "wand_max_abs_err_mm": (wand_max_err if np.isfinite(wand_max_err) else ""),
        "pass": case_pass,
    }
    return point_rows, wand_rows, case_row


def main():
    if not ROOT.exists():
        raise RuntimeError(f"Root not found: {ROOT}")

    cases = sorted([p for p in ROOT.glob("case_*") if p.is_dir()], key=lambda p: p.name)

    out_points = ROOT / "verification_geometry_points.csv"
    out_wand = ROOT / "verification_wand_length.csv"
    out_cases = ROOT / "verification_geometry_cases.csv"
    out_summary = ROOT / "verification_geometry_summary.txt"

    all_points = []
    all_wand = []
    all_cases = []

    for case_dir in cases:
        point_rows, wand_rows, case_row = verify_case(case_dir, THRESH_MM)
        all_points.extend(point_rows)
        all_wand.extend(wand_rows)
        all_cases.append(case_row)
        print(
            f"[{case_row['case']}] {case_row['status']} "
            f"total={case_row['total_points']} failed={case_row['failed_points']} "
            f"wand_failed={case_row['wand_failed_frames']} pass={case_row['pass']}"
        )

    with out_points.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "case", "frame", "point_idx", "num_obs", "num_valid_rays", "invalid_los",
                "max_residual_mm", "pass", "reason"
            ],
        )
        w.writeheader()
        for r in all_points:
            w.writerow(r)

    with out_wand.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["case", "frame", "wand_abs_err_mm", "pass", "reason"],
        )
        w.writeheader()
        for r in all_wand:
            w.writerow(r)

    with out_cases.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "case", "status", "reason", "total_points", "failed_points", "max_err_mm",
                "wand_failed_frames", "wand_max_abs_err_mm", "pass"
            ],
        )
        w.writeheader()
        for r in all_cases:
            w.writerow(r)

    n_total = len(all_cases)
    n_ok = sum(1 for r in all_cases if r["status"] == "OK")
    n_skip = sum(1 for r in all_cases if r["status"] != "OK")
    n_pass = sum(1 for r in all_cases if r["status"] == "OK" and r["pass"])
    pts_total = sum(int(r["total_points"]) for r in all_cases if r["status"] == "OK")
    pts_fail = sum(int(r["failed_points"]) for r in all_cases if r["status"] == "OK")

    wand_total = len(all_wand)
    wand_failed = sum(1 for r in all_wand if not bool(r["pass"]))

    lines = [
        f"root: {ROOT}",
        f"threshold_mm: {THRESH_MM}",
        f"wand_len_mm: {WAND_LEN_MM}",
        f"wand_threshold_mm: {WAND_THRESH_MM}",
        f"cases_total: {n_total}",
        f"cases_ok: {n_ok}",
        f"cases_skip: {n_skip}",
        f"cases_pass_all_points: {n_pass}",
        f"points_total: {pts_total}",
        f"points_failed: {pts_fail}",
        f"wand_total_frames: {wand_total}",
        f"wand_failed_frames: {wand_failed}",
        f"overall_pass: {pts_fail == 0 and wand_failed == 0 and n_ok > 0}",
        "",
        f"points_csv: {out_points}",
        f"wand_csv: {out_wand}",
        f"cases_csv: {out_cases}",
    ]
    out_summary.write_text("\n".join(lines), encoding="utf-8")

    print("\n==== DONE ====")
    print(f"Summary: {out_summary}")
    print(f"Points : {out_points}")
    print(f"Wand   : {out_wand}")
    print(f"Cases  : {out_cases}")


if __name__ == "__main__":
    main()


[case_001] OK total=4000 failed=0 wand_failed=0 pass=True
[case_002] OK total=4000 failed=0 wand_failed=0 pass=True
[case_003] OK total=4000 failed=0 wand_failed=0 pass=True
[case_004] OK total=4000 failed=0 wand_failed=0 pass=True
[case_005] OK total=4000 failed=0 wand_failed=0 pass=True
[case_006] OK total=4000 failed=0 wand_failed=0 pass=True
[case_007] OK total=4000 failed=0 wand_failed=0 pass=True
[case_008] OK total=4000 failed=0 wand_failed=0 pass=True
[case_009] OK total=4000 failed=0 wand_failed=0 pass=True
[case_010] OK total=4000 failed=0 wand_failed=0 pass=True
[case_011] OK total=4000 failed=0 wand_failed=0 pass=True
[case_012] OK total=4000 failed=0 wand_failed=0 pass=True
[case_013] OK total=4000 failed=0 wand_failed=0 pass=True
[case_014] OK total=4000 failed=0 wand_failed=0 pass=True
[case_015] OK total=4000 failed=0 wand_failed=0 pass=True
[case_016] OK total=4000 failed=0 wand_failed=0 pass=True
[case_017] OK total=4000 failed=0 wand_failed=0 pass=True
[case_018] OK 